# NumPy — Numerical Computing in Python

NumPy is the library almost every other data library in Python is built on. Pandas stores its
columns as NumPy arrays. Matplotlib plots NumPy arrays. Scikit-learn trains on NumPy arrays.
So learning NumPy properly is not one topic among many — it is the foundation for the rest.

### What this notebook covers

- Why plain Python lists run out of steam for numerical work
- Arrays: creating them, their shape, their dtype, and what that actually means in memory
- Indexing, slicing, reshaping, and the view-versus-copy trap that catches everyone once
- Element-wise math, aggregations, and the `axis` argument (the single most misunderstood thing in NumPy)
- Broadcasting, from concrete examples up to the general rule
- Boolean masking and fancy indexing
- Combining and splitting arrays, sorting and searching
- Linear algebra and random number generation
- Vectorisation: what it buys you, and when it does not
- A mini project analysing student exam results end to end

### Prerequisites

You should already be comfortable with Python variables, `for` loops, `if` statements, functions,
lists and dictionaries. Nothing beyond that is assumed. Where we use something slightly clever —
a list comprehension, `zip`, `enumerate`, unpacking — it gets explained on the spot.

### How the notebook is written

Most topics are shown twice. First the way you would naturally write it with the Python you
already know: an explicit loop, an intermediate variable, one step per line. Then the way
someone who works with NumPy every day would write it. The second version is not there to show
off; the point is that you can read the first one and understand exactly what the second one does.
Occasionally the "clever" version is genuinely worse, and when that happens the notebook says so.

## Setup

Run the next cell only if NumPy is not already installed. If `import numpy` works in the cell
after it, skip this one.

In [1]:
# %pip install numpy

In [2]:
import numpy as np

print(np.__version__)

2.2.1


`import numpy as np` is the universal convention. Every book, tutorial and codebase uses `np`,
so do the same — code that says `import numpy as numpy_library` will read as strange to anyone else.

The examples below were written against NumPy 2.x. If your version prints something starting
with `1.`, nearly everything still works, and the few places where the modern API differs are
flagged as we go.

## Why not just use lists?

A Python list can hold numbers, so why bother with a new type? Try the most basic numerical
task there is: adding two sets of measurements together.

Say we recorded the morning and evening temperature (in °C) for five days in a greenhouse.

In [3]:
morning = [18.5, 19.2, 17.8, 20.1, 18.9]
evening = [24.3, 25.1, 23.6, 26.4, 24.8]

morning + evening

[18.5, 19.2, 17.8, 20.1, 18.9, 24.3, 25.1, 23.6, 26.4, 24.8]

That is probably not what you wanted. For lists, `+` means *concatenate* — stick the second
list onto the end of the first. There is no way to tell Python "I meant add them pairwise",
because a list is a general-purpose container. It has no idea it is holding numbers.

To actually add them element by element you have to write the loop yourself.

### Beginner approach — an explicit loop

In [4]:
daily_average = []

for i in range(len(morning)):
    daily_average.append((morning[i] + evening[i]) / 2)

print(daily_average)

[21.4, 22.15, 20.700000000000003, 23.25, 21.85]


Walk through what happens:

- `daily_average = []` creates an empty list to collect results into.
- `range(len(morning))` produces the positions `0, 1, 2, 3, 4`.
- On each pass, `morning[i]` and `evening[i]` pull out the two readings for the same day.
- `(... + ...) / 2` averages them, and `append` puts that value on the end of the result list.

It works, and it is perfectly readable. The problem is that this pattern — loop over positions,
compute, append — reappears for every single operation. Want the difference between evening and
morning? Another loop. Want to convert to Fahrenheit? Another loop. Want only the days above 20°C?
Another loop. Your program becomes mostly loop scaffolding, and the actual arithmetic gets buried.

### The NumPy approach

In [5]:
morning_arr = np.array([18.5, 19.2, 17.8, 20.1, 18.9])
evening_arr = np.array([24.3, 25.1, 23.6, 26.4, 24.8])

daily_average = (morning_arr + evening_arr) / 2
print(daily_average)

[21.4  22.15 20.7  23.25 21.85]


`np.array(...)` converts a list into a NumPy array (its real type name is `ndarray`,
"n-dimensional array"). Once the data is in an array, `+` no longer means concatenate — it means
"add matching elements", and `/ 2` means "divide every element by 2".

This is the central idea of NumPy, and it has a name: **element-wise**, or **vectorised**,
operations. You describe what should happen to the whole collection, and the loop happens inside
NumPy's compiled C code instead of in your Python source.

Now the follow-up questions become one line each:

In [6]:
swing = evening_arr - morning_arr
fahrenheit = morning_arr * 9 / 5 + 32

print("Daily swing:       ", swing)
print("Morning in °F:     ", np.round(fahrenheit, 1))
print("Warmest morning:   ", morning_arr.max())
print("Mean daily average:", daily_average.mean())

Daily swing:        [5.8 5.9 5.8 6.3 5.9]
Morning in °F:      [65.3 66.6 64.  68.2 66. ]
Warmest morning:    20.1
Mean daily average: 21.869999999999997


### What is different under the hood

The reason lists cannot do this is worth understanding, because it explains most of NumPy's
other behaviour too.

A Python list is an array of *pointers*. Each slot holds a memory address pointing at a full
Python object somewhere else in memory. A Python `float` object carries a type tag, a reference
count and the actual 8 bytes of numeric data. So a list of five floats is five scattered objects
plus a table of addresses, and adding two of them means five separate trips to fetch objects,
five type checks, five additions, five new objects.

A NumPy array is one flat block of memory holding just the raw numbers, all of the same type,
back to back. The array object stores the numbers' type once, plus the shape. Adding two arrays
becomes a tight loop over contiguous memory with no type checking and no object creation.

Two practical consequences follow immediately:

1. Arrays are **homogeneous** — every element has the same type. You cannot have one string in
   the middle of an array of integers without changing the type of the whole array.
2. Arrays have a **fixed size**. There is no `append` that grows an array in place, because that
   would mean reallocating the whole block. Growing an array means building a new one.

In [7]:
mixed = np.array([1, 2, 3.5])
print(mixed, mixed.dtype)

with_text = np.array([1, 2, "three"])
print(with_text, with_text.dtype)

[1.  2.  3.5] float64
['1' '2' 'three'] <U21


In the first case NumPy promoted the integers to floats so that all three values share one type.
In the second, the only type that can hold both numbers and text is a string type, so `1` and `2`
became the strings `'1'` and `'2'`. That `<U21` means "Unicode strings up to 21 characters".

This silent promotion is a common source of confusion: if arithmetic on an array suddenly fails
or gives strange results, check `dtype` first.

## The mental model

Hold on to this picture for the rest of the notebook:

```text
An array = one flat buffer of same-typed numbers
           +  a shape that says how to read it as a grid
```

The same twelve numbers in memory can be read as a flat list of 12, a 3×4 table, a 4×3 table,
or a 2×2×3 cube. Nothing moves; only the shape description changes.

```text
buffer:  1  2  3  4  5  6  7  8  9 10 11 12

shape (12,)      →  1 2 3 4 5 6 7 8 9 10 11 12

shape (3, 4)     →  1  2  3  4
                    5  6  7  8
                    9 10 11 12

shape (2, 2, 3)  →  block 0:  1  2  3      block 1:  7  8  9
                              4  5  6                10 11 12
```

Once "shape is just an interpretation" clicks, reshaping, transposing and broadcasting all stop
feeling like magic.

## Array attributes

Every array can describe itself. These are the attributes you will actually use, and checking
them is the first thing to do when array code misbehaves.

In [8]:
scores = np.array([[72, 85, 90, 64],
                   [88, 79, 95, 70],
                   [60, 71, 68, 55]])

print(scores)
print("ndim     :", scores.ndim)
print("shape    :", scores.shape)
print("size     :", scores.size)
print("dtype    :", scores.dtype)
print("itemsize :", scores.itemsize, "bytes per element")
print("nbytes   :", scores.nbytes, "bytes total")

[[72 85 90 64]
 [88 79 95 70]
 [60 71 68 55]]
ndim     : 2
shape    : (3, 4)
size     : 12
dtype    : int64
itemsize : 8 bytes per element
nbytes   : 96 bytes total


- **`ndim`** — how many dimensions (axes). `1` for a plain sequence, `2` for a table, `3` for a stack of tables.
- **`shape`** — a tuple with the length along each axis. `(3, 4)` means 3 rows and 4 columns. The
  length of this tuple always equals `ndim`.
- **`size`** — total number of elements, the product of the shape. Here 3 × 4 = 12.
- **`dtype`** — the single element type shared by everything in the array.
- **`itemsize`** and **`nbytes`** — bytes per element, and bytes overall. `int64` is 64 bits = 8 bytes,
  so 12 elements occupy 96 bytes.

One detail about `shape` for one-dimensional arrays: it prints as `(5,)`, with a trailing comma.
That comma is Python's syntax for a one-element tuple, not a typo and not a hidden second
dimension. `(5,)` is genuinely different from `(5, 1)` and from `(1, 5)`, and mixing those up is
behind a good share of shape errors later on.

In [9]:
one_d = np.array([10, 20, 30, 40, 50])
row = np.array([[10, 20, 30, 40, 50]])
column = np.array([[10], [20], [30], [40], [50]])

for name, arr in [("one_d", one_d), ("row", row), ("column", column)]:
    print(f"{name:7} ndim={arr.ndim}  shape={arr.shape}")

one_d   ndim=1  shape=(5,)
row     ndim=2  shape=(1, 5)
column  ndim=2  shape=(5, 1)


The loop above uses two small pieces of ordinary Python worth naming, since they appear
constantly from here on:

- A list of `(name, value)` tuples is unpacked directly in the `for` statement.
  `for name, arr in [...]` assigns both parts of each tuple in one step, which beats writing
  `item[0]` and `item[1]`.
- `f"{name:7}"` is an f-string with a *width* specifier: pad `name` to 7 characters so the columns
  line up. Nothing NumPy-specific, just readable output.

### Building 2D and 3D arrays by hand

A 2D array is a list of equal-length lists. A 3D array is a list of equal-shaped 2D arrays.
The nesting in your source code mirrors the nesting of the shape.

In [10]:
# Three sensors, four hourly readings each, across two days.
readings = np.array([
    [[21.0, 21.4, 22.1, 22.8],
     [20.6, 20.9, 21.5, 22.0],
     [21.8, 22.2, 22.9, 23.4]],

    [[19.9, 20.3, 21.0, 21.6],
     [19.4, 19.8, 20.4, 21.1],
     [20.7, 21.1, 21.8, 22.3]],
])

print("shape:", readings.shape, "→ (days, sensors, hours)")
print("sensor 2 on day 0:", readings[0, 2])

shape: (2, 3, 4) → (days, sensors, hours)
sensor 2 on day 0: [21.8 22.2 22.9 23.4]


Read the shape `(2, 3, 4)` from the outside in: 2 days, each holding 3 sensors, each holding
4 readings. When you index `readings[0, 2]` you are saying "day 0, sensor 2", and what is left
over is that sensor's 4 hourly readings.

Get into the habit of writing shapes down like that — `(days, sensors, hours)` — in a comment or
a variable name. Shapes carry meaning, but the array itself does not remember the meaning, so
you have to.

### One mistake to avoid immediately

Rows of different lengths do not make a 2D array.

In [11]:
try:
    ragged = np.array([[1, 2, 3], [4, 5]])
except ValueError as e:
    print("ValueError:", e)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.


The notebook catches the error on purpose so every cell still runs top to bottom, but the message
is the real one NumPy produces:

```text
setting an array element with a sequence. The requested array has an inhomogeneous shape
after 1 dimensions. The detected shape was (2,) + inhomogeneous part.
```

Translated: "you promised a rectangle and handed me something jagged". A rectangular block is
exactly what allows one flat buffer, so the restriction is not arbitrary. If your data genuinely
is jagged, keep it as a list of arrays, or pad the short rows to equal length.

## Creating arrays without typing them out

Literal values are fine for four numbers and hopeless for four thousand. These constructors cover
almost everything you need in practice.

In [12]:
print("zeros(5)         ", np.zeros(5))
print("ones(4, int)     ", np.ones(4, dtype=int))
print("full(4, 7.5)     ", np.full(4, 7.5))
print("arange(0, 10, 2) ", np.arange(0, 10, 2))
print("linspace(0, 1, 5)", np.linspace(0, 1, 5))

zeros(5)          [0. 0. 0. 0. 0.]
ones(4, int)      [1 1 1 1]
full(4, 7.5)      [7.5 7.5 7.5 7.5]
arange(0, 10, 2)  [0 2 4 6 8]
linspace(0, 1, 5) [0.   0.25 0.5  0.75 1.  ]


- **`np.zeros(shape)`** and **`np.ones(shape)`** — filled with 0.0 or 1.0. Both default to
  `float64`; pass `dtype=int` when you want whole numbers. `zeros` is the normal way to
  pre-allocate space you are about to fill.
- **`np.full(shape, value)`** — the same idea with any fill value.
- **`np.arange(start, stop, step)`** — like the built-in `range`, but returns an array and accepts
  fractional steps. As with `range`, `stop` is **excluded**.
- **`np.linspace(start, stop, count)`** — `count` evenly spaced values, and here `stop`
  **is included**. Reach for this whenever you care about how many points you get rather than the
  gap between them; it is the standard way to build an x-axis for plotting.

That inconsistency — `arange` excludes the endpoint, `linspace` includes it — is genuinely
confusing and purely historical. Say the difference out loud once and it tends to stick.

In [13]:
print("arange(0, 1, 0.25):", np.arange(0, 1, 0.25))
print("linspace(0, 1, 5) :", np.linspace(0, 1, 5))

arange(0, 1, 0.25): [0.   0.25 0.5  0.75]
linspace(0, 1, 5) : [0.   0.25 0.5  0.75 1.  ]


Be careful with `np.arange` and fractional steps. A step like 0.1 has no exact binary
representation, so the values it produces are very slightly off, and that leaks out in ways that
surprise people.

In [14]:
stepped = np.arange(0, 0.7, 0.1)
spaced = np.linspace(0, 0.6, 7)

print("arange  :", stepped)
print("linspace:", spaced)
print("does arange contain exactly 0.6?  ", (stepped == 0.6).any())
print("does linspace contain exactly 0.6?", (spaced == 0.6).any())

arange  : [0.  0.1 0.2 0.3 0.4 0.5 0.6]
linspace: [0.  0.1 0.2 0.3 0.4 0.5 0.6]
does arange contain exactly 0.6?   False
does linspace contain exactly 0.6? True


Both print `0.6`. Only one of them actually holds 0.6 — the `arange` value is
0.6000000000000001, and printing rounds it for display. Any code that tests for equality against
0.6 will quietly disagree with what you see on screen.

The same inexactness can even let a value reach the endpoint that was supposed to be excluded:

In [15]:
odd = np.arange(0, 2.25, 0.009)
print("last value:", odd[-1], "but stop was 2.25 and should be excluded")
print("is the last value >= stop?", odd[-1] >= 2.25)

last value: 2.25 but stop was 2.25 and should be excluded
is the last value >= stop? True


So with fractional steps, neither "the values are exactly what I typed" nor "the endpoint is
excluded" is reliable. Keep `arange` for integer steps, where it is exact and reads well, and use
`linspace` whenever the values are fractional or the count matters.

### Two-dimensional constructors

In [16]:
print(np.zeros((2, 3)))
print()
print(np.eye(3))
print()
print(np.full((2, 4), 9))

[[0. 0. 0.]
 [0. 0. 0.]]

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

[[9 9 9 9]
 [9 9 9 9]]


Note the double parentheses in `np.zeros((2, 3))`. The function takes *one* argument, the shape,
and the shape is the tuple `(2, 3)`. Writing `np.zeros(2, 3)` passes two separate arguments and
fails, because the second positional parameter of `zeros` is `dtype`, not another dimension.

`np.eye(n)` builds the identity matrix — ones on the diagonal, zeros elsewhere. In linear algebra
it plays the role the number 1 plays in ordinary arithmetic.

### `np.empty`, and why you usually do not want it

In [17]:
junk = np.empty(4)
print(junk)

[0.25 0.5  0.75 1.  ]


`np.empty` reserves the memory without setting the values, so you see whatever bytes happened to
be sitting there. Your output will differ from mine, and it may well look like plausible numbers,
which is exactly what makes it dangerous.

It exists because skipping initialisation is marginally faster, and that only matters if you are
about to overwrite every single element. In ordinary code use `np.zeros`. A bug caused by
forgetting to fill one slot costs far more than the microseconds saved.

### Random arrays

Random data is useful for testing and for teaching examples, and NumPy's modern interface for it
is a **Generator** object.

In [18]:
rng = np.random.default_rng(42)

print("uniform [0, 1) :", np.round(rng.random(4), 3))
print("integers 1-6   :", rng.integers(1, 7, size=5))
print("normal(70, 10) :", np.round(rng.normal(70, 10, size=5), 1))

uniform [0, 1) : [0.774 0.439 0.859 0.697]
integers 1-6   : [2 1 4 6 5]
normal(70, 10) : [66.8 69.8 61.5 78.8 77.8]


`np.random.default_rng(42)` creates a generator seeded with 42. Seeding makes the "random" numbers
reproducible: run this notebook tomorrow, on a different machine, and the same values come out.
That matters more than it sounds — reproducible examples let you compare your output against the
notebook's, and reproducible tests fail for real reasons rather than by luck.

The three methods used above:

- `rng.random(n)` — floats in `[0, 1)`.
- `rng.integers(low, high, size)` — whole numbers with `high` **excluded**, so `1, 7` is a die roll.
- `rng.normal(mean, sd, size)` — a bell curve with the given centre and spread.

You will also meet the older style in books and Stack Overflow answers:

```python
np.random.seed(42)
np.random.rand(4)
np.random.randint(1, 7, 5)
```

That still works, but it mutates a single hidden global generator shared by your entire program,
including library code you did not write. A `Generator` object is self-contained, so two parts of
a program can each hold their own and never interfere. Prefer `default_rng` in new code, and be
able to read the old form, because plenty of it is still out there.

## Exercises — arrays and creation

**Level 1.** Create an array of the ten even numbers from 2 to 20 inclusive, then print its
`shape`, `dtype` and sum.

**Level 2.** Build a 3×5 array in which every value is 7, without typing 7 fifteen times. Then
build a 3×5 array whose rows are `1..5`, `6..10` and `11..15`, using `arange` and `.reshape(3, 5)`
(reshaping gets its own section shortly, but the name says what it does).

**Level 3.** `np.linspace(0, 1, 11)` and `np.arange(0, 1.1, 0.1)` look like they should produce
the same eleven values. Print both, subtract one from the other, and explain what you see.
Which would you use to build an x-axis for a plot, and why?

Solutions to every exercise set are at the end of the notebook. Attempt them first — reading a
solution feels like understanding, and it is not the same thing.

## Indexing and slicing

For one-dimensional arrays, indexing works exactly like lists: zero-based positions, negative
numbers count from the end, and `start:stop:step` slices exclude `stop`.

We will use a week of rainfall measurements (mm) for the examples.

In [19]:
rainfall = np.array([0.0, 12.4, 3.1, 0.0, 28.7, 6.2, 1.5])
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

print("first day      :", rainfall[0])
print("last day       :", rainfall[-1])
print("second to last :", rainfall[-2])
print("Tue to Thu     :", rainfall[1:4])
print("from Fri on    :", rainfall[4:])
print("every other day:", rainfall[::2])
print("reversed       :", rainfall[::-1])

first day      : 0.0
last day       : 1.5
second to last : 6.2
Tue to Thu     : [12.4  3.1  0. ]
from Fri on    : [28.7  6.2  1.5]
every other day: [ 0.   3.1 28.7  1.5]
reversed       : [ 1.5  6.2 28.7  0.   3.1 12.4  0. ]


A couple of reminders about slice syntax, since misreading it causes off-by-one bugs:

- `rainfall[1:4]` gives positions 1, 2 and 3 — **four is where it stops, not what it includes**.
  Think of the numbers as marking the boundaries between elements rather than the elements themselves.
- Leaving a side blank means "all the way": `[4:]` is "from 4 to the end", `[:3]` is "start up to 3",
  `[:]` is everything.
- The third number is the step. `[::2]` takes every second element; `[::-1]` walks backwards,
  which is the standard way to reverse a sequence in Python.

### Where array indexing goes beyond lists

Two capabilities have no list equivalent at all, and both get their own section later. Worth
seeing now so you know they exist:

In [20]:
print("pick specific positions:", rainfall[[0, 4, 6]])
print("pick by condition      :", rainfall[rainfall > 5])

pick specific positions: [ 0.  28.7  1.5]
pick by condition      : [12.4 28.7  6.2]


The first passes a *list of indices* and gets those elements back, in that order. The second
passes a *condition* and gets back only the elements where it held. Try either with a plain
Python list and you get a `TypeError`.

### Two-dimensional indexing

Here is a small gradebook: 4 students down the rows, 3 subjects across the columns.

In [21]:
marks = np.array([[78, 82, 91],
                  [65, 70, 58],
                  [88, 94, 79],
                  [55, 61, 72]])

students = ["Aarti", "Bilal", "Chirag", "Divya"]
subjects = ["Maths", "Physics", "Chemistry"]

print(marks)
print("shape:", marks.shape, "→ (students, subjects)")

[[78 82 91]
 [65 70 58]
 [88 94 79]
 [55 61 72]]
shape: (4, 3) → (students, subjects)


In [22]:
print("Bilal's physics mark   :", marks[1, 1])
print("Chirag's chemistry mark:", marks[2, 2])
print("Divya's first subject  :", marks[3, 0])

Bilal's physics mark   : 70
Chirag's chemistry mark: 79
Divya's first subject  : 55


`marks[row, column]` — one pair of brackets, a comma inside. This is the biggest syntactic
difference from nested lists, where you would write `marks[1][1]`.

Both spellings work on a NumPy array, and they are not equally good:

In [23]:
print(marks[1, 1], "and", marks[1][1])

70 and 70


`marks[1][1]` does two operations: extract row 1 as a new array object, then index into it.
`marks[1, 1]` goes straight to the element. On a tiny array the difference is irrelevant, but
the comma form is what NumPy code looks like, and it is the only form that lets you slice both
dimensions at once — which is where the real power is.

### Selecting whole rows and columns

In [24]:
print("row 0 (Aarti's marks)  :", marks[0])
print("row 0, spelled fully   :", marks[0, :])
print("column 1 (all physics) :", marks[:, 1])

row 0 (Aarti's marks)  : [78 82 91]
row 0, spelled fully   : [78 82 91]
column 1 (all physics) : [82 70 94 61]


Read the colon as "everything along this axis".

- `marks[0, :]` — row 0, all columns.
- `marks[:, 1]` — all rows, column 1.
- `marks[0]` is shorthand for `marks[0, :]`. Missing trailing indices are filled in with `:`.

Selecting a column is where nested lists become genuinely painful. Compare:

In [25]:
marks_as_lists = [[78, 82, 91], [65, 70, 58], [88, 94, 79], [55, 61, 72]]

physics_beginner = []
for row in marks_as_lists:
    physics_beginner.append(row[1])

physics_pro = marks[:, 1]

print("loop over lists:", physics_beginner)
print("NumPy column   :", physics_pro)

loop over lists: [82, 70, 94, 61]
NumPy column   : [82 70 94 61]


The list version has to visit every row and pull out one item, because a list of lists is stored
row by row and has no concept of a column. NumPy knows the shape, so "all rows, column 1" is a
thing you can simply ask for.

Notice also what comes back: a **1D array of length 4**, not a 4×1 column. NumPy drops any axis
you index with a single number. Only slices preserve the dimension.

In [26]:
print("marks[:, 1]    shape:", marks[:, 1].shape)
print("marks[:, 1:2]  shape:", marks[:, 1:2].shape)

marks[:, 1]    shape: (4,)
marks[:, 1:2]  shape: (4, 1)


`1` means "that one item, axis gone". `1:2` means "a slice that happens to contain one item, axis
kept". This distinction matters when you start combining arrays and the shapes have to agree.

### Slicing both dimensions

In [27]:
print("first two students, first two subjects:")
print(marks[0:2, 0:2])

print("\nall students, last two subjects:")
print(marks[:, 1:])

print("\nevery other student, reversed subjects:")
print(marks[::2, ::-1])

first two students, first two subjects:
[[78 82]
 [65 70]]

all students, last two subjects:
[[82 91]
 [70 58]
 [94 79]
 [61 72]]

every other student, reversed subjects:
[[91 82 78]
 [79 94 88]]


The rule is simple once you see it: each position between the commas is an independent slice
applied to its own axis. `marks[0:2, 0:2]` means "rows 0 and 1, and within those, columns 0 and 1".

### Assignment through a slice

Indexing also works on the left of an `=`, and it applies to the whole selection at once.

In [28]:
attendance = np.zeros((4, 5), dtype=int)

attendance[0] = 1
attendance[:, 4] = 1
attendance[2, 1:3] = 9

print(attendance)

[[1 1 1 1 1]
 [0 0 0 0 1]
 [0 9 9 0 1]
 [0 0 0 0 1]]


- `attendance[0] = 1` filled row 0 with ones. One scalar spreads across the whole selection —
  that is broadcasting, which we cover properly soon.
- `attendance[:, 4] = 1` filled the last column.
- `attendance[2, 1:3] = 9` filled just two cells in row 2.

This is how you edit arrays. There is no `append`, and you rarely loop to assign.

### The dtype trap when assigning

In [29]:
counts = np.array([10, 20, 30])
counts[0] = 7.9
print(counts, counts.dtype)

[ 7 20 30] int64


The 7.9 became 7, with no warning. The array's dtype is `int64`, every slot holds an integer, and
7.9 does not fit, so NumPy truncated it.

This is the most common silent data-loss bug in NumPy code. If a column holds measurements rather
than counts, create it as float from the start:

In [30]:
measurements = np.array([10, 20, 30], dtype=float)
measurements[0] = 7.9
print(measurements, measurements.dtype)

[ 7.9 20.  30. ] float64


You can also convert an existing array with `astype`, which returns a **new** array rather than
changing the original:

In [31]:
counts_as_float = counts.astype(float)
print("converted:", counts_as_float, counts_as_float.dtype)
print("original :", counts, counts.dtype)

converted: [ 7. 20. 30.] float64
original : [ 7 20 30] int64


## Views versus copies

This section is short, and skipping it will cost you an afternoon of debugging one day.

**A basic slice of a NumPy array does not copy the data. It is a view onto the same memory.**

In [32]:
original = np.array([10, 20, 30, 40, 50])
chunk = original[1:4]

print("chunk before:", chunk)

chunk[0] = 999

print("chunk after :", chunk)
print("original    :", original)

chunk before: [20 30 40]
chunk after : [999  30  40]
original    : [ 10 999  30  40  50]


We never mentioned `original` on the line that assigned 999, and it changed anyway. That is
because `chunk` is not a new array holding copied values; it is a window on positions 1 to 3 of
`original`'s buffer. Writing through the window writes to the buffer.

With a Python list, slicing copies, so the same code leaves the original alone:

In [33]:
original_list = [10, 20, 30, 40, 50]
chunk_list = original_list[1:4]
chunk_list[0] = 999

print("list slice :", chunk_list)
print("list source:", original_list)

list slice : [999, 30, 40]
list source: [10, 20, 30, 40, 50]


Why does NumPy behave differently? Because arrays are meant to be big. If slicing a
one-gigabyte array copied it, the innocent-looking `big[1:-1]` would double your memory use.
Views make slicing free, and the cost is that you have to know when you are holding one.

### Checking, and taking a real copy

In [34]:
view = original[1:4]
copy = original[1:4].copy()

print("view shares memory:", np.shares_memory(original, view))
print("copy shares memory:", np.shares_memory(original, copy))
print("view.base is original:", view.base is original)
print("copy.base:", copy.base)

view shares memory: True
copy shares memory: False
view.base is original: True
copy.base: None


- `np.shares_memory(a, b)` answers the question directly and is the tool to reach for when you
  suspect aliasing.
- `.base` points at the array a view borrows from, and is `None` for an array that owns its data.
- `.copy()` gives you an independent array. Use it whenever you intend to modify a slice and
  keep the original intact.

### Which operations give views, and which give copies

In [35]:
grid = np.arange(12).reshape(3, 4)

checks = {
    "grid[1:, 2:]  (basic slice)": grid[1:, 2:],
    "grid.T        (transpose)  ": grid.T,
    "grid.reshape(4, 3)         ": grid.reshape(4, 3),
    "grid.ravel()               ": grid.ravel(),
    "grid.flatten()             ": grid.flatten(),
    "grid[[0, 2]]  (index list) ": grid[[0, 2]],
    "grid[grid > 5] (condition) ": grid[grid > 5],
}

for label, result in checks.items():
    print(f"{label} → view" if np.shares_memory(grid, result) else f"{label} → copy")

grid[1:, 2:]  (basic slice) → view
grid.T        (transpose)   → view
grid.reshape(4, 3)          → view
grid.ravel()                → view
grid.flatten()              → copy
grid[[0, 2]]  (index list)  → copy
grid[grid > 5] (condition)  → copy


The pattern is worth remembering as a rule of thumb:

| Operation | Result |
| --- | --- |
| basic slicing (`a[1:4]`, `a[:, 2]`) | **view** |
| `reshape`, `ravel`, `T`, `newaxis` | **view** when possible |
| `flatten()` | always a copy |
| indexing with a list or array of positions | copy |
| indexing with a boolean condition | copy |
| `astype`, arithmetic, `np.where`, `concatenate` | copy (a new array) |

`ravel` and `flatten` both flatten to 1D and differ only in this: `ravel` hands back a view when
the memory layout allows one, `flatten` always copies. So `flatten` is the safe default and
`ravel` is the cheap one.

### The bug this causes in real code

In [36]:
def normalise_broken(data):
    """Scale a column to 0-1. Looks harmless, quietly edits the caller's array."""
    column = data[:, 0]
    column -= column.min()
    column /= column.max()
    return column


table = np.array([[10.0, 1.0], [20.0, 2.0], [30.0, 3.0]])
print("before:\n", table)

normalise_broken(table)

print("after:\n", table)

before:
 [[10.  1.]
 [20.  2.]
 [30.  3.]]
after:
 [[0.  1. ]
 [0.5 2. ]
 [1.  3. ]]


The function was asked to return a normalised column and it also rewrote the input. `data[:, 0]`
is a view, and `-=` and `/=` are **in-place** operators, so both wrote straight into `table`.

The fix is one word:

In [37]:
def normalise(data):
    column = data[:, 0].copy()
    column -= column.min()
    column /= column.max()
    return column


table = np.array([[10.0, 1.0], [20.0, 2.0], [30.0, 3.0]])
result = normalise(table)

print("returned:", result)
print("input untouched:\n", table)

returned: [0.  0.5 1. ]
input untouched:
 [[10.  1.]
 [20.  2.]
 [30.  3.]]


The more idiomatic version avoids in-place operators entirely. `column - column.min()` builds a
new array rather than modifying anything, so there is nothing to alias:

In [38]:
def normalise_clean(data, col=0):
    column = data[:, col]
    return (column - column.min()) / (column.max() - column.min())


table = np.array([[10.0, 1.0], [20.0, 2.0], [30.0, 3.0]])
print("returned:", normalise_clean(table))
print("input untouched:\n", table)

returned: [0.  0.5 1. ]
input untouched:
 [[10.  1.]
 [20.  2.]
 [30.  3.]]


That is the general lesson: `x = x - 1` creates a new array and is always safe; `x -= 1` writes
into whatever buffer `x` points at and is safe only if you own that buffer. Prefer the former
unless you have a reason — usually memory — to want the latter.

## Reshaping

Reshaping changes how the buffer is interpreted. The element count has to stay the same.

In [39]:
flat = np.arange(1, 13)
print("flat       :", flat, flat.shape)

as_3x4 = flat.reshape(3, 4)
print("\nreshape(3, 4):")
print(as_3x4)

as_4x3 = flat.reshape(4, 3)
print("\nreshape(4, 3):")
print(as_4x3)

flat       : [ 1  2  3  4  5  6  7  8  9 10 11 12] (12,)

reshape(3, 4):
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]

reshape(4, 3):
[[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]


NumPy fills the new shape in **row-major** order by default: fill the last axis first, then step
forward in the one before it. Practically, it reads the buffer left to right and lays it out row
by row. That is why `1 2 3 4` became the first row of the 3×4 version and `1 2 3` the first row
of the 4×3 version.

In [40]:
try:
    flat.reshape(5, 3)
except ValueError as e:
    print("ValueError:", e)

ValueError: cannot reshape array of size 12 into shape (5,3)


12 elements cannot be arranged as 5×3 = 15. The message — `cannot reshape array of size 12 into
shape (5,3)` — is one you will see often, and it always means the same thing: check `arr.size`
against the product of your target shape.

### Letting NumPy work out one dimension

In [41]:
print(flat.reshape(3, -1).shape)
print(flat.reshape(-1, 2).shape)
print(flat.reshape(2, 2, -1).shape)

(3, 4)
(6, 2)
(2, 2, 3)


`-1` means "you figure this one out from the total". `reshape(3, -1)` says "three rows, however
many columns that needs". Only one `-1` is allowed, because with two there would be many answers.

This is used constantly, particularly `reshape(-1, 1)` to turn a flat array into a single column,
which many libraries expect.

### Flattening back down

In [42]:
grid = np.arange(6).reshape(2, 3)

print("ravel   :", grid.ravel())
print("flatten :", grid.flatten())
print("reshape :", grid.reshape(-1))

ravel   : [0 1 2 3 4 5]
flatten : [0 1 2 3 4 5]
reshape : [0 1 2 3 4 5]


All three produce the same values. As covered above, `flatten` always copies while `ravel` and
`reshape(-1)` return views when they can. When in doubt, use `flatten`: a needless copy of a
small array costs nothing, whereas an accidental view costs you a debugging session.

### Transposing

In [43]:
print("original (4 students × 3 subjects):")
print(marks)

print("\ntransposed (3 subjects × 4 students):")
print(marks.T)

print("\nshapes:", marks.shape, "→", marks.T.shape)

original (4 students × 3 subjects):
[[78 82 91]
 [65 70 58]
 [88 94 79]
 [55 61 72]]

transposed (3 subjects × 4 students):
[[78 65 88 55]
 [82 70 94 61]
 [91 58 79 72]]

shapes: (4, 3) → (3, 4)


`.T` flips rows and columns. `marks.T` is the same data seen as "subjects down the side, students
across the top", and it is a view — no data was moved, only the description of how to walk the
buffer.

Transposing is how you line up arrays whose axes mean the right things but sit the wrong way round.
`.T` is shorthand for `np.transpose(arr)`; for arrays with more than two dimensions,
`np.transpose` lets you specify the exact axis order, which occasionally matters (image data is
the usual example, where you swap between "height, width, channels" and "channels, height, width").

### Adding and removing axes

Sometimes an operation needs a `(3, 1)` column where you have a `(3,)` flat array. Three ways to
insert an axis, all equivalent:

In [44]:
values = np.array([5, 10, 15])

print("original           :", values.shape)
print("values[:, None]    :", values[:, None].shape)
print("expand_dims axis=1 :", np.expand_dims(values, axis=1).shape)
print("reshape(-1, 1)     :", values.reshape(-1, 1).shape)

print("\nas a column:")
print(values[:, None])

original           : (3,)
values[:, None]    : (3, 1)
expand_dims axis=1 : (3, 1)
reshape(-1, 1)     : (3, 1)

as a column:
[[ 5]
 [10]
 [15]]


`values[:, None]` reads as "all the original elements along the first axis, and a new axis of
length 1 after it". `None` in an index position means "insert an axis here"; you may also see it
written `np.newaxis`, which is literally the same object with a clearer name:

In [45]:
print(values[np.newaxis, :].shape, "row")
print(values[:, np.newaxis].shape, "column")

(1, 3) row
(3, 1) column


The reverse operation, `np.squeeze`, removes axes of length 1.

In [46]:
awkward = np.array([[[1], [2], [3]]])
print("awkward shape :", awkward.shape)
print("squeezed shape:", np.squeeze(awkward).shape)

awkward shape : (1, 3, 1)
squeezed shape: (3,)


Shapes like `(1, 3, 1)` usually appear as a by-product of some other operation, and `squeeze` is
the broom. Be a little careful with it in library code: if an array might legitimately have a
length-1 axis, `squeeze` will silently remove that too. `reshape` with an explicit shape says
what you mean.

## Exercises — indexing, views and reshaping

**Level 1.** From `temps = np.array([18, 21, 25, 30, 28, 22, 19])`, print the first three values,
the last two values, every second value, and the values in reverse.

**Level 2.** Build `table = np.arange(1, 21).reshape(4, 5)`. Extract the second row, the last
column, the 2×2 block in the bottom-right corner, and the sub-table consisting of rows 0 and 2
only (all columns).

**Level 3.** Write a function `top_row_sum(matrix)` that returns the sum of the largest row of a
2D array, and guarantee that the caller's array is never modified. Then deliberately write a
version that *does* modify its input through a view, and demonstrate the difference with
`np.shares_memory`.

## Mathematical operations

Arithmetic between arrays is element-wise, and arithmetic between an array and a single number
applies that number to every element.

In [47]:
units_sold = np.array([120, 85, 200, 65, 150])
unit_price = np.array([25.0, 40.0, 12.5, 80.0, 30.0])

revenue = units_sold * unit_price
after_tax = revenue * 0.82

print("revenue  :", revenue)
print("after tax:", np.round(after_tax, 2))

revenue  : [3000. 3400. 2500. 5200. 4500.]
after tax: [2460. 2788. 2050. 4264. 3690.]


`units_sold * unit_price` multiplies position 0 by position 0, position 1 by position 1, and so on.
It is **not** matrix multiplication — that is `@`, covered in the linear algebra section. `*`
between arrays always means "pair them up and multiply".

### The mathematical functions

NumPy provides its own versions of the usual maths functions, and they work on whole arrays.
These are called **ufuncs** (universal functions).

In [48]:
values = np.array([1.0, 4.0, 9.0, 16.0, 25.0])

print("sqrt  :", np.sqrt(values))
print("log   :", np.round(np.log(values), 3))
print("exp   :", np.round(np.exp([0, 1, 2]), 3))
print("power :", np.power(values, 0.5))
print("abs   :", np.abs([-3, 4, -5]))
print("round :", np.round([1.234, 5.678], 1))
print("floor :", np.floor([1.7, -1.7]))
print("ceil  :", np.ceil([1.2, -1.2]))

sqrt  : [1. 2. 3. 4. 5.]
log   : [0.    1.386 2.197 2.773 3.219]
exp   : [1.    2.718 7.389]
power : [1. 2. 3. 4. 5.]
abs   : [3 4 5]
round : [1.2 5.7]
floor : [ 1. -2.]
ceil  : [ 2. -1.]


Use `np.sqrt(arr)` rather than Python's `math.sqrt(arr)` — the `math` module only handles single
numbers and will raise a `TypeError` on an array. Anything in `math` has a NumPy counterpart that
works element-wise.

### Aggregations

An aggregation collapses many numbers into one.

In [49]:
scores = np.array([72, 85, 90, 64, 88, 79, 95, 70, 60, 71])

print("sum      :", scores.sum())
print("mean     :", scores.mean())
print("median   :", np.median(scores))
print("std      :", round(scores.std(), 2))
print("var      :", round(scores.var(), 2))
print("min, max :", scores.min(), scores.max())
print("range    :", np.ptp(scores))
print("argmin   :", scores.argmin(), "→ value", scores[scores.argmin()])
print("argmax   :", scores.argmax(), "→ value", scores[scores.argmax()])

sum      : 774
mean     : 77.4
median   : 75.5
std      : 11.17
var      : 124.84
min, max : 60 95
range    : 35
argmin   : 8 → value 60
argmax   : 6 → value 95


Most of these read as you would expect. Three deserve comment:

- **`np.ptp`** is "peak to peak", i.e. `max - min`. (In NumPy 2.x it is only available as a
  function; the old `scores.ptp()` method was removed.)
- **`argmin` / `argmax`** return the **position** of the smallest or largest value, not the value
  itself. The `arg` prefix means "argument", as in "the index at which". This is how you answer
  "*which* student scored highest" rather than "what was the highest score":

In [50]:
students = np.array(["Aarti", "Bilal", "Chirag", "Divya", "Esha"])
final_marks = np.array([78, 92, 65, 88, 71])

best = final_marks.argmax()
print(f"Top scorer: {students[best]} with {final_marks[best]}")

Top scorer: Bilal with 92


- **`std`** is the standard deviation. Note that NumPy divides by `n` by default (the *population*
  standard deviation), while Pandas divides by `n - 1` (the *sample* version). Two libraries can
  therefore report slightly different numbers for the same data, which is alarming the first time
  you notice it. Pass `ddof=1` to make NumPy match:

In [51]:
print("NumPy default (ddof=0):", round(scores.std(), 4))
print("Sample std    (ddof=1):", round(scores.std(ddof=1), 4))

NumPy default (ddof=0): 11.1732
Sample std    (ddof=1): 11.7776


### Cumulative operations

Cumulative functions return an array of the same length, where each element is the running result
up to that point.

In [52]:
daily_sales = np.array([120, 95, 140, 180, 75, 210, 160])

print("daily     :", daily_sales)
print("cumulative:", np.cumsum(daily_sales))
print("cumulative max:", np.maximum.accumulate(daily_sales))

daily     : [120  95 140 180  75 210 160]
cumulative: [120 215 355 535 610 820 980]
cumulative max: [120 120 140 180 180 210 210]


`np.cumsum` answers "how much had we sold by the end of each day". `np.cumprod` does the same with
multiplication, which is how you compound growth rates. `np.maximum.accumulate` gives the
running best-so-far, useful for "record high" style questions.

Compare to how you would write a running total without NumPy:

In [53]:
running = []
total = 0
for value in daily_sales:
    total += value
    running.append(total)

print("loop  :", running)
print("cumsum:", list(np.cumsum(daily_sales)))

loop  : [np.int64(120), np.int64(215), np.int64(355), np.int64(535), np.int64(610), np.int64(820), np.int64(980)]
cumsum: [np.int64(120), np.int64(215), np.int64(355), np.int64(535), np.int64(610), np.int64(820), np.int64(980)]


The loop is clear enough, and worth writing once so you know what `cumsum` does. After that
there is no reason to write it again.

## The `axis` argument

This is the concept students get wrong most often, so it is worth going slowly.

When an array has more than one dimension, "the sum" is ambiguous. Sum of everything? Sum of each
row? Sum of each column? The `axis` argument picks which.

Our example: quarterly sales for three products.

In [54]:
sales = np.array([[120, 135, 150, 160],
                  [ 90,  85, 110, 100],
                  [200, 210, 190, 230]])

products = ["Keyboard", "Mouse", "Monitor"]
quarters = ["Q1", "Q2", "Q3", "Q4"]

print(sales)
print("shape:", sales.shape, "→ (products, quarters)")

[[120 135 150 160]
 [ 90  85 110 100]
 [200 210 190 230]]
shape: (3, 4) → (products, quarters)


```text
             Q1    Q2    Q3    Q4
Keyboard    120   135   150   160      ← axis 1 runs this way (across columns)
Mouse        90    85   110   100
Monitor     200   210   190   230
              ↑
        axis 0 runs this way (down rows)
```

The rule that actually works, and is worth memorising:

> **`axis=n` is the axis that disappears.** NumPy collapses that axis and keeps the rest.

Our shape is `(3, 4)`.

- `axis=0` collapses the 3 → result has shape `(4,)`, one number per **quarter**.
- `axis=1` collapses the 4 → result has shape `(3,)`, one number per **product**.

In [55]:
print("sales.sum()       =", sales.sum(), " (everything, no axis)")
print("sales.sum(axis=0) =", sales.sum(axis=0), " shape", sales.sum(axis=0).shape)
print("sales.sum(axis=1) =", sales.sum(axis=1), " shape", sales.sum(axis=1).shape)

sales.sum()       = 1780  (everything, no axis)
sales.sum(axis=0) = [410 430 450 490]  shape (4,)
sales.sum(axis=1) = [565 385 830]  shape (3,)


Now attach meaning to those numbers:

In [56]:
per_quarter = sales.sum(axis=0)
per_product = sales.sum(axis=1)

for quarter, value in zip(quarters, per_quarter):
    print(f"{quarter}: {value} units across all products")

print()
for product, value in zip(products, per_product):
    print(f"{product:9}: {value} units across the year")

Q1: 410 units across all products
Q2: 430 units across all products
Q3: 450 units across all products
Q4: 490 units across all products

Keyboard : 565 units across the year
Mouse    : 385 units across the year
Monitor  : 830 units across the year


`zip(quarters, per_quarter)` pairs up two sequences element by element, so the loop variable
`quarter` walks the labels while `value` walks the numbers. It stops at the shorter of the two,
which is usually what you want and occasionally hides a bug — if the lengths disagree, `zip`
silently ignores the extra.

### The trick for remembering which axis is which

Students often try to memorise "axis 0 is rows, axis 1 is columns" and then get confused, because
`axis=0` gives you a result *per column*. Both statements are true and that is the problem.

Use the disappearing-axis rule instead. Say the shape out loud:

```text
sales.shape == (3, 4)
                ↑  ↑
          axis 0    axis 1
```

`axis=0` eats the 3. Whatever was indexed by that axis — products — is gone, summed away. What
survives is the quarters.

If you cannot remember, check the shape of the result. It tells you immediately what you got.

In [57]:
for axis in (None, 0, 1):
    result = sales.mean(axis=axis)
    print(f"axis={str(axis):4} → shape {np.shape(result)}  values {np.round(result, 1)}")

axis=None → shape ()  values 148.3
axis=0    → shape (4,)  values [136.7 143.3 150.  163.3]
axis=1    → shape (3,)  values [141.2  96.2 207.5]


### Keeping the dimension

Sometimes you want the collapsed axis kept as length 1 so the result still lines up with the
original for arithmetic. That is `keepdims=True`.

In [58]:
row_totals = sales.sum(axis=1)
row_totals_kept = sales.sum(axis=1, keepdims=True)

print("without keepdims:", row_totals.shape)
print(row_totals)
print("\nwith keepdims:", row_totals_kept.shape)
print(row_totals_kept)

without keepdims: (3,)
[565 385 830]

with keepdims: (3, 1)
[[565]
 [385]
 [830]]


Why bother? Because this then works directly:

In [59]:
share_of_year = sales / sales.sum(axis=1, keepdims=True)
print(np.round(share_of_year * 100, 1))

[[21.2 23.9 26.5 28.3]
 [23.4 22.1 28.6 26. ]
 [24.1 25.3 22.9 27.7]]


Each row now shows what percentage of that product's annual sales fell in each quarter, and the
rows sum to 100. The `(3, 1)` shape aligns against the `(3, 4)` array row by row. Without
`keepdims` you would have a `(3,)` array, which NumPy tries to align against the *columns*, and
you would get either wrong answers or a shape error. The next section explains exactly why.

### Axis beyond two dimensions

The same rule scales. Recall the sensor array with shape `(days, sensors, hours)`:

In [60]:
readings = np.array([
    [[21.0, 21.4, 22.1, 22.8], [20.6, 20.9, 21.5, 22.0], [21.8, 22.2, 22.9, 23.4]],
    [[19.9, 20.3, 21.0, 21.6], [19.4, 19.8, 20.4, 21.1], [20.7, 21.1, 21.8, 22.3]],
])

print("shape                       :", readings.shape)
print("mean(axis=0) per sensor/hour:", readings.mean(axis=0).shape)
print("mean(axis=1) per day/hour   :", readings.mean(axis=1).shape)
print("mean(axis=2) per day/sensor :", readings.mean(axis=2).shape)
print("mean(axis=(0, 2)) per sensor:", readings.mean(axis=(0, 2)).shape)

print("\naverage per sensor across both days and all hours:")
print(np.round(readings.mean(axis=(0, 2)), 2))

shape                       : (2, 3, 4)
mean(axis=0) per sensor/hour: (3, 4)
mean(axis=1) per day/hour   : (2, 4)
mean(axis=2) per day/sensor : (2, 3)
mean(axis=(0, 2)) per sensor: (3,)

average per sensor across both days and all hours:
[21.26 20.71 22.02]


`axis=(0, 2)` collapses two axes at once: average over days and over hours, leaving one number per
sensor. Reading it as "which axes disappear" makes even this case easy.

### Common mistakes with axis

In [61]:
try:
    sales.sum(axis=2)
except np.exceptions.AxisError as e:
    print("AxisError:", e)

AxisError: axis 2 is out of bounds for array of dimension 2


`axis=2` on a 2D array: there is no third axis. The message
`axis 2 is out of bounds for array of dimension 2` is unusually clear. Valid axes for a 2D array
are 0 and 1 (and the negatives `-1`, `-2`, counting from the end — `axis=-1` means "the last
axis", which is a nice way to say "across each row" regardless of how many dimensions there are).

The mistake that hurts more is using the wrong axis and getting plausible-looking numbers:

In [62]:
print("Average per product (correct):", sales.mean(axis=1).round(1))
print("Average per quarter (also valid, different question):", sales.mean(axis=0).round(1))

Average per product (correct): [141.2  96.2 207.5]
Average per quarter (also valid, different question): [136.7 143.3 150.  163.3]


Both run. Both produce sensible-looking numbers. Only one answers your question. There is no error
message for asking the wrong thing, so check the shape and the length of the result: three
products, four quarters. If you expected three numbers and got four, you used the wrong axis.

## Broadcasting

Broadcasting is the set of rules NumPy uses when the two arrays in an operation have different
shapes. You have already used it several times without being told.

In [63]:
temperatures = np.array([18.5, 21.0, 25.3, 30.1])

print(temperatures + 2)
print(temperatures * 1.8 + 32)

[20.5 23.  27.3 32.1]
[65.3  69.8  77.54 86.18]


`temperatures + 2` adds a single number to a four-element array. Strictly, "add" needs two things
of the same shape, so NumPy behaves *as if* the 2 were stretched into `[2, 2, 2, 2]`. No such
array is actually built — that is the point, it would be a waste of memory — but the result is
what you would get if it had been.

That is broadcasting: **when shapes do not match, stretch the smaller one along axes of length 1
until they do.**

### A genuinely useful case

Back to the gradebook. Each subject is marked out of a different total, and we want percentages.

In [64]:
marks = np.array([[78, 82, 91],
                  [65, 70, 58],
                  [88, 94, 79],
                  [55, 61, 72]])

max_marks = np.array([100, 120, 150])

print("marks shape    :", marks.shape)
print("max_marks shape:", max_marks.shape)

percentages = marks / max_marks * 100
print("\npercentages:")
print(np.round(percentages, 1))

marks shape    : (4, 3)
max_marks shape: (3,)

percentages:
[[78.  68.3 60.7]
 [65.  58.3 38.7]
 [88.  78.3 52.7]
 [55.  50.8 48. ]]


A `(4, 3)` array was divided by a `(3,)` array. NumPy lined the shapes up from the right:

```text
marks      (4, 3)
max_marks     (3,)      →  treated as (1, 3)
result     (4, 3)
```

The length-3 axis matched. The missing first axis was filled in as 1 and then stretched to 4,
so every row was divided by the same three subject totals. Exactly what we wanted, and no loop.

The beginner version of the same thing:

In [65]:
percentages_loop = np.zeros_like(marks, dtype=float)

for student in range(marks.shape[0]):
    for subject in range(marks.shape[1]):
        percentages_loop[student, subject] = marks[student, subject] / max_marks[subject] * 100

print(np.round(percentages_loop, 1))
print("same result:", np.allclose(percentages_loop, percentages))

[[78.  68.3 60.7]
 [65.  58.3 38.7]
 [88.  78.3 52.7]
 [55.  50.8 48. ]]
same result: True


Two nested loops, two index variables, and one line of arithmetic buried inside. It is not wrong,
and writing it once is a good way to be sure you understand what the broadcast version does.

`np.zeros_like(marks, dtype=float)` creates an array with the same shape as `marks` but filled
with zeros and typed as float — that `dtype=float` matters, because without it the results would
be truncated to integers.

`np.allclose(a, b)` checks whether two float arrays agree to within a tiny tolerance. Use it
rather than `a == b` when comparing floats, because `0.1 + 0.2 == 0.3` is `False` in every
language that uses binary floating point.

In [66]:
print("0.1 + 0.2 == 0.3       :", 0.1 + 0.2 == 0.3)
print("np.isclose(0.1+0.2, 0.3):", np.isclose(0.1 + 0.2, 0.3))

0.1 + 0.2 == 0.3       : False
np.isclose(0.1+0.2, 0.3): True


### The rules, stated properly

Line the two shapes up **from the right**. For each position:

1. If the lengths are equal → fine.
2. If one of them is 1 → that one is stretched to match the other.
3. Otherwise → error.

A missing dimension on the left counts as 1.

```text
Works:
  (4, 3)  and  (3,)      →  (3,) becomes (1, 3), stretched to (4, 3)      ✓
  (4, 3)  and  (4, 1)    →  the 1 stretches to 3                          ✓
  (4, 1)  and  (1, 3)    →  both stretch                    → (4, 3)      ✓
  (2, 3, 4) and (4,)     →  aligns with the last axis                     ✓

Fails:
  (4, 3)  and  (4,)      →  3 vs 4, neither is 1                          ✗
  (4, 3)  and  (2, 3)    →  4 vs 2, neither is 1                          ✗
```

In [67]:
row_vector = np.array([[1, 2, 3]])
col_vector = np.array([[10], [20], [30], [40]])

print("row shape:", row_vector.shape, " col shape:", col_vector.shape)
print("\nsum broadcasts to", (col_vector + row_vector).shape, ":")
print(col_vector + row_vector)

row shape: (1, 3)  col shape: (4, 1)

sum broadcasts to (4, 3) :
[[11 12 13]
 [21 22 23]
 [31 32 33]
 [41 42 43]]


A `(4, 1)` plus a `(1, 3)` gives a `(4, 3)` grid, because *both* arrays get stretched: the column
repeats across 3 columns, the row repeats down 4 rows. This is how you build a multiplication
table, a distance matrix, or any "every combination of A and B" calculation without loops.

In [68]:
sizes = np.array([1, 2, 3, 4, 5])
print(sizes[:, None] * sizes[None, :])

[[ 1  2  3  4  5]
 [ 2  4  6  8 10]
 [ 3  6  9 12 15]
 [ 4  8 12 16 20]
 [ 5 10 15 20 25]]


`sizes[:, None]` is the `(5, 1)` column and `sizes[None, :]` is the `(1, 5)` row, so the product
is the 5×5 multiplication table. This is worth staring at for a moment, because the pattern
"turn one array into a column, the other into a row, combine" is how a surprising number of
problems get solved without loops.

### When broadcasting fails

In [69]:
try:
    marks + np.array([1, 2, 3, 4])
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (4,3) (4,) 


`marks` is `(4, 3)` and the other array is `(4,)`. Aligning from the right gives 3 against 4:
not equal, neither is 1, so it fails with
`operands could not be broadcast together with shapes (4,3) (4,)`.

The intent was probably "add one number per student", which means the extra array has to be a
**column**, not a row. Fix it by giving it the shape that says so:

In [70]:
bonus_per_student = np.array([1, 2, 3, 4])

print(marks + bonus_per_student[:, None])

[[79 83 92]
 [67 72 60]
 [91 97 82]
 [59 65 76]]


`bonus_per_student[:, None]` reshapes `(4,)` to `(4, 1)`, which aligns against the rows. Every
student's bonus is added across all three of their subjects.

This is the single most common broadcasting fix: an array whose length matches the *wrong* axis
needs `[:, None]` to point it at the right one.

### A practical pattern: centring each column

Subtract each column's mean from that column, a routine first step before many statistical methods.

In [71]:
data = np.array([[10.0, 200.0, 3.0],
                 [12.0, 180.0, 5.0],
                 [ 9.0, 220.0, 4.0],
                 [11.0, 190.0, 6.0]])

column_means = data.mean(axis=0)
centred = data - column_means

print("column means:", column_means)
print("\ncentred:")
print(centred)
print("\nnew column means (should be ~0):", centred.mean(axis=0).round(12))

column means: [ 10.5 197.5   4.5]

centred:
[[ -0.5   2.5  -1.5]
 [  1.5 -17.5   0.5]
 [ -1.5  22.5  -0.5]
 [  0.5  -7.5   1.5]]

new column means (should be ~0): [0. 0. 0.]


`data.mean(axis=0)` gives a `(3,)` array of column means, which broadcasts across the rows
automatically — no `keepdims` needed here, because aligning from the right already matches the
columns.

Compare with centring each **row**, where you do need to be explicit:

In [72]:
row_means = data.mean(axis=1, keepdims=True)
print("row means shape:", row_means.shape)
print(np.round(data - row_means, 2))

row means shape: (4, 1)
[[-61.   129.   -68.  ]
 [-53.67 114.33 -60.67]
 [-68.67 142.33 -73.67]
 [-58.   121.   -63.  ]]


The rule of thumb: when you aggregate over `axis=0` the result already aligns for broadcasting;
when you aggregate over `axis=1` you almost always want `keepdims=True`.

## Exercises — maths, axis and broadcasting

**Level 1.** Given `prices = np.array([250, 480, 120, 890, 310])`, compute the total, the mean,
the most expensive price, and the position of the cheapest item.

**Level 2.** Using the `sales` array from the axis section (3 products × 4 quarters), find the
best quarter for each product (as a quarter name, not an index), and the product with the highest
annual total. Use `argmax` with a sensible axis.

**Level 3.** You have `distances = np.array([5.0, 12.5, 3.2, 8.8])` in km and
`fuel_rates = np.array([0.08, 0.11, 0.15])` in litres per km for three vehicles. Build a table of
fuel needed with one row per vehicle and one column per trip, using broadcasting and no loops.
Then state what the shape of the result must be, and why.

## Boolean masking

Comparing an array to a value gives you an array of `True`/`False`, one per element. That array
can then be used to select elements. This is how filtering is done in NumPy, and the same idea
carries straight over to Pandas.

In [73]:
rainfall = np.array([0.0, 12.4, 3.1, 0.0, 28.7, 6.2, 1.5])

is_wet = rainfall > 5
print("condition:", is_wet)
print("dtype    :", is_wet.dtype)

condition: [False  True False False  True  True False]
dtype    : bool


`rainfall > 5` did not return one answer; it compared every element and returned seven answers.
The result is a **boolean mask** — the same shape as the original, with `True` wherever the
condition held.

Put that mask inside the brackets and you get the matching elements:

In [74]:
print("wet days only:", rainfall[is_wet])
print("same thing in one step:", rainfall[rainfall > 5])

wet days only: [12.4 28.7  6.2]
same thing in one step: [12.4 28.7  6.2]


`rainfall[rainfall > 5]` is the form you will write in practice. Read it inside out: the inner
part builds the mask, the outer part selects with it.

The result is shorter than the original (only 3 of 7 days qualified) and it is a **copy**, not a
view — NumPy cannot express "these scattered positions" as a window on the buffer.

### Beginner version, for comparison

In [75]:
wet_days = []

for value in rainfall:
    if value > 5:
        wet_days.append(value)

print(wet_days)

[np.float64(12.4), np.float64(28.7), np.float64(6.2)]


The loop says the same thing: look at each value, keep it if it passes. Once you have seen that
`rainfall[rainfall > 5]` is exactly this, the short version stops being mysterious.

A middle option exists too, and it is worth knowing because it is plain Python:

In [76]:
print([value for value in rainfall if value > 5])

[np.float64(12.4), np.float64(28.7), np.float64(6.2)]


```text
[value for value in rainfall if value > 5]
 ↑         ↑                    ↑
 what      where it comes       which ones
 to keep   from                 to keep
```

A list comprehension reads left to right as "collect `value`, for each `value` in `rainfall`,
where `value > 5`". It is shorter than the loop and works on any iterable, not just arrays.
For NumPy data, though, prefer the mask: it keeps the result as an array, and it does the work
in compiled code rather than one Python step per element.

### Counting and testing without extracting

Because `True` counts as 1 and `False` as 0, summing a mask counts the matches.

In [77]:
print("number of wet days :", (rainfall > 5).sum())
print("fraction of wet days:", (rainfall > 5).mean().round(3))
print("any day over 25mm? :", (rainfall > 25).any())
print("every day rained?  :", (rainfall > 0).all())

number of wet days : 3
fraction of wet days: 0.429
any day over 25mm? : True
every day rained?  : False


- `.sum()` on a mask → how many.
- `.mean()` on a mask → what proportion (the mean of 0s and 1s is the fraction of 1s).
- `.any()` → was there at least one?
- `.all()` → did every element qualify?

`(rainfall > 5).sum()` is a genuinely nice idiom. The beginner equivalent is a counter variable
and an `if` inside a loop; this is one expression and reads almost like the question.

### Combining conditions

In [78]:
temps = np.array([18.5, 21.0, 25.3, 30.1, 27.6, 19.8, 33.2])

comfortable = (temps > 20) & (temps < 30)
extreme = (temps < 19) | (temps > 32)

print("temps       :", temps)
print("comfortable :", temps[comfortable])
print("extreme     :", temps[extreme])
print("not comfortable:", temps[~comfortable])

temps       : [18.5 21.  25.3 30.1 27.6 19.8 33.2]
comfortable : [21.  25.3 27.6]
extreme     : [18.5 33.2]
not comfortable: [18.5 30.1 19.8 33.2]


Three things to get right here, and all three trip people up:

**Use `&`, `|`, `~` — not `and`, `or`, `not`.** The word forms ask for a single true/false answer
about the whole array, and NumPy refuses to guess:

In [79]:
try:
    result = (temps > 20) and (temps < 30)
except ValueError as e:
    print("ValueError:", e)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


`The truth value of an array with more than one element is ambiguous` means exactly what it says:
you asked "is this array true?" and there is no sensible answer for seven values at once. The
symbol forms `&`, `|` and `~` work element by element and return a mask, which is what you want.

**Parenthesise each condition.** `&` binds more tightly than `>` in Python, so without parentheses
`temps > 20 & temps < 30` is parsed as `temps > (20 & temps) < 30`, which is nonsense and usually
errors out. The parentheses are not optional stylistic noise.

**`~` is "not".** `~comfortable` flips every `True` to `False`, giving the days that were not
comfortable.

### `np.where` — choose per element

`np.where(condition, value_if_true, value_if_false)` is the array equivalent of an `if`/`else`
applied to every element.

In [80]:
labels = np.where(temps > 30, "hot", "fine")
print(labels)

adjusted = np.where(temps > 30, temps - 5, temps)
print(adjusted)

['fine' 'fine' 'fine' 'hot' 'fine' 'fine' 'hot']
[18.5 21.  25.3 25.1 27.6 19.8 28.2]


The second example says "cap the extreme readings by subtracting 5, leave the rest alone". All
three arguments are arrays (or broadcastable to arrays), and the choice is made position by position.

Nesting `np.where` gives you an `elif` chain, but it gets unreadable quickly:

In [81]:
category = np.where(temps > 30, "hot", np.where(temps > 20, "warm", "cool"))
print(category)

['cool' 'warm' 'warm' 'hot' 'warm' 'cool' 'hot']


Two levels are tolerable. Beyond that use `np.select`, which takes a list of conditions and a
matching list of results, and reads much more like the table of rules it represents:

In [82]:
conditions = [temps > 30, temps > 25, temps > 20]
choices = ["hot", "warm", "mild"]

print(np.select(conditions, choices, default="cool"))

['cool' 'mild' 'warm' 'hot' 'warm' 'cool' 'hot']


`np.select` checks the conditions in order and takes the first match, so put the most specific
condition first. The `default` covers anything that matched nothing.

### Called with one argument, `np.where` gives positions

In [83]:
positions = np.where(temps > 25)
print("positions:", positions)
print("as a plain array:", positions[0])
print("values there:", temps[positions])

positions: (array([2, 3, 4, 6]),)
as a plain array: [2 3 4 6]
values there: [25.3 30.1 27.6 33.2]


That returned a *tuple* containing one array, which looks odd until you see the 2D case: for an
n-dimensional array, `np.where` returns n arrays, one per axis, so that the coordinates pair up.

In [84]:
marks = np.array([[78, 82, 91],
                  [65, 70, 58],
                  [88, 94, 79],
                  [55, 61, 72]])

rows, cols = np.where(marks < 60)
print("rows:", rows, "cols:", cols)

for r, c in zip(rows, cols):
    print(f"mark {marks[r, c]} at student {r}, subject {c}")

rows: [1 3] cols: [2 0]
mark 58 at student 1, subject 2
mark 55 at student 3, subject 0


`rows, cols = np.where(...)` unpacks the tuple into two names. Reading them together gives the
coordinates of each failing mark. If you only need the values, `marks[marks < 60]` is simpler; use
the position form when you need to know *where*.

### Assigning through a mask

In [85]:
readings = np.array([21.5, -999.0, 22.1, 23.4, -999.0, 22.8])

cleaned = readings.copy()
cleaned[cleaned == -999] = np.nan

print("raw    :", readings)
print("cleaned:", cleaned)
print("mean ignoring missing:", np.nanmean(cleaned).round(3))
print("plain mean           :", cleaned.mean())

raw    : [  21.5 -999.    22.1   23.4 -999.    22.8]
cleaned: [21.5  nan 22.1 23.4  nan 22.8]
mean ignoring missing: 22.45
plain mean           : nan


`-999` is a common sentinel for "sensor failed". Replacing it with `np.nan` (not-a-number) marks
those entries as genuinely missing rather than as an absurd temperature.

Note the two means. `cleaned.mean()` returns `nan`, because any arithmetic involving `nan` produces
`nan` — this is deliberate, and it is a feature: it stops missing data from silently disappearing
into your results. When you do want to skip the gaps, use the `nan`-aware versions:
`np.nanmean`, `np.nansum`, `np.nanmax`, `np.nanstd` and friends.

Two more things about `nan` worth knowing now:

In [86]:
print("nan == nan:", np.nan == np.nan)
print("isnan     :", np.isnan(cleaned))
print("how many missing:", np.isnan(cleaned).sum())

nan == nan: False
isnan     : [False  True False False  True False]
how many missing: 2


`nan` is not equal to itself, so `arr == np.nan` never finds anything. Always use `np.isnan(arr)`.
And `nan` only exists for float arrays — assigning it into an integer array raises an error,
which is another reason measurements belong in float columns.

## Fancy indexing

Indexing with a list or array of positions pulls out exactly those elements, in exactly that order,
and repeats are allowed.

In [87]:
products = np.array(["keyboard", "mouse", "monitor", "webcam", "headset"])
stock = np.array([34, 120, 8, 45, 67])

wanted = [2, 0, 4]
print("selected products:", products[wanted])
print("their stock      :", stock[wanted])
print("with a repeat    :", products[[1, 1, 3]])

selected products: ['monitor' 'keyboard' 'headset']
their stock      : [ 8 34 67]
with a repeat    : ['mouse' 'mouse' 'webcam']


This is how you reorder or sample an array. Combined with `argsort` (next section), it is also how
you sort one array by another — a pattern that comes up constantly.

For 2D arrays, passing one list selects rows:

In [88]:
print("students 0 and 2:")
print(marks[[0, 2]])

print("\nspecific cells (student 0 subject 2, student 3 subject 1):")
print(marks[[0, 3], [2, 1]])

students 0 and 2:
[[78 82 91]
 [88 94 79]]

specific cells (student 0 subject 2, student 3 subject 1):
[91 61]


The second form pairs the two lists up: `(0, 2)` and `(3, 1)`. It returns a flat array of those
two individual cells, *not* a 2×2 block. If you wanted the block — rows 0 and 3 crossed with
columns 2 and 1 — you have to say so, by making the row selector a column:

In [89]:
print(marks[[[0], [3]], [2, 1]])

[[91 82]
 [72 61]]


That is broadcasting again, now applied to index arrays: a `(2, 1)` row selector against a `(2,)`
column selector gives a `(2, 2)` result. It is clever, and it is also the point where fancy
indexing starts costing more to read than it saves. `marks[np.ix_([0, 3], [2, 1])]` says the same
thing more legibly:

In [90]:
print(marks[np.ix_([0, 3], [2, 1])])

[[91 82]
 [72 61]]


## Sorting and searching

In [91]:
scores = np.array([72, 95, 64, 88, 79, 60, 91])
names = np.array(["Aarti", "Bilal", "Chirag", "Divya", "Esha", "Farhan", "Gita"])

print("sorted values     :", np.sort(scores))
print("descending        :", np.sort(scores)[::-1])
print("original untouched:", scores)

sorted values     : [60 64 72 79 88 91 95]
descending        : [95 91 88 79 72 64 60]
original untouched: [72 95 64 88 79 60 91]


`np.sort(arr)` returns a **new** sorted array. The method form `arr.sort()` sorts **in place** and
returns `None` — a distinction inherited from Python's `sorted()` versus `list.sort()`. Assigning
`x = arr.sort()` and wondering why `x` is `None` is a rite of passage.

There is no `descending=True` option; reverse the result with `[::-1]`.

### `argsort` — sorting one array by another

In [92]:
order = np.argsort(scores)
print("argsort       :", order)
print("scores sorted :", scores[order])
print("names in score order:", names[order])

argsort       : [5 2 0 4 3 6 1]
scores sorted : [60 64 72 79 88 91 95]
names in score order: ['Farhan' 'Chirag' 'Aarti' 'Esha' 'Divya' 'Gita' 'Bilal']


`np.argsort` returns the **positions** that would sort the array. Here it starts with `5`, meaning
"the smallest score is at position 5" (Farhan's 60).

Feed those positions into another array of the same length and you have sorted that array by the
first one. This is the standard way to answer "who are the top three?":

In [93]:
top_three = np.argsort(scores)[::-1][:3]

for rank, i in enumerate(top_three, start=1):
    print(f"{rank}. {names[i]:7} {scores[i]}")

1. Bilal   95
2. Gita    91
3. Divya   88


Reading `np.argsort(scores)[::-1][:3]` in pieces: sort positions ascending, reverse them so the
best is first, keep the first three.

`enumerate(top_three, start=1)` walks the array while also counting, starting the count at 1 so
the ranks read naturally. Without `start=1` you would get 0, 1, 2 and have to add one by hand.

### Unique values and counts

In [94]:
grades = np.array(["B", "A", "C", "B", "A", "B", "D", "A"])

print("unique:", np.unique(grades))

values, counts = np.unique(grades, return_counts=True)
for value, count in zip(values, counts):
    print(f"{value}: {count}")

unique: ['A' 'B' 'C' 'D']
A: 3
B: 3
C: 1
D: 1


`np.unique` returns the distinct values, sorted. With `return_counts=True` it also returns how
many times each appeared, as a second array — this is the NumPy equivalent of tallying with a
dictionary, and Pandas' `value_counts()` does the same job with nicer output.

The beginner version, for comparison:

In [95]:
tally = {}
for grade in grades:
    if grade in tally:
        tally[grade] += 1
    else:
        tally[grade] = 1

print(tally)

{np.str_('B'): 3, np.str_('A'): 3, np.str_('C'): 1, np.str_('D'): 1}


### Other searching tools

In [96]:
print("clip to 70-90     :", np.clip(scores, 70, 90))
print("index for insertion:", np.searchsorted(np.sort(scores), 80))
print("is 88 present?    :", np.isin(88, scores))
print("which are in a set?:", np.isin(grades, ["A", "B"]))

clip to 70-90     : [72 90 70 88 79 70 90]
index for insertion: 4
is 88 present?    : True
which are in a set?: [ True  True False  True  True  True False  True]


- `np.clip(arr, low, high)` squashes everything outside the range to the boundary. Handy for
  capping outliers.
- `np.searchsorted(sorted_arr, value)` says where `value` would go to keep the array sorted —
  the basis of binary search, and much faster than scanning for large sorted data.
- `np.isin(values, collection)` tests membership element-wise. (Older code uses `np.in1d`, which
  still works but is discouraged in favour of `isin`.)

## Combining and splitting arrays

In [97]:
first_half = np.array([1, 2, 3])
second_half = np.array([4, 5, 6])

print("concatenate:", np.concatenate([first_half, second_half]))
print("vstack:")
print(np.vstack([first_half, second_half]))
print("hstack:", np.hstack([first_half, second_half]))
print("stack (new axis):")
print(np.stack([first_half, second_half]))

concatenate: [1 2 3 4 5 6]
vstack:
[[1 2 3]
 [4 5 6]]
hstack: [1 2 3 4 5 6]
stack (new axis):
[[1 2 3]
 [4 5 6]]


These four look similar and differ in exactly one respect: whether a **new axis** is created.

- `np.concatenate` joins along an axis that already exists. Two `(3,)` arrays give a `(6,)` array.
- `np.vstack` stacks vertically, treating 1D arrays as rows. Two `(3,)` give `(2, 3)`.
- `np.hstack` joins horizontally, which for 1D arrays is the same as `concatenate`.
- `np.stack` always adds a new axis. Two `(3,)` give `(2, 3)`, like `vstack` here, but `stack`
  lets you choose where the new axis goes with `axis=`.

In [98]:
print("stack axis=0 shape:", np.stack([first_half, second_half], axis=0).shape)
print("stack axis=1 shape:", np.stack([first_half, second_half], axis=1).shape)
print("stack axis=1:")
print(np.stack([first_half, second_half], axis=1))

stack axis=0 shape: (2, 3)
stack axis=1 shape: (3, 2)
stack axis=1:
[[1 4]
 [2 5]
 [3 6]]


For 2D arrays, `concatenate` with an explicit axis is the clearest choice:

In [99]:
top = np.array([[1, 2, 3], [4, 5, 6]])
bottom = np.array([[7, 8, 9]])
side = np.array([[10], [20]])

print("stacked vertically (axis=0):")
print(np.concatenate([top, bottom], axis=0))

print("\njoined horizontally (axis=1):")
print(np.concatenate([top, side], axis=1))

stacked vertically (axis=0):
[[1 2 3]
 [4 5 6]
 [7 8 9]]

joined horizontally (axis=1):
[[ 1  2  3 10]
 [ 4  5  6 20]]


The shapes have to agree on every axis except the one you are joining along. Adding a row means
the column counts must match; adding a column means the row counts must match. When it fails,
print both shapes — the mismatch is always obvious once you see them side by side.

In [100]:
try:
    np.concatenate([top, np.array([[1, 2]])], axis=0)
except ValueError as e:
    print("ValueError:", e)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 3 and the array at index 1 has size 2


`np.column_stack` deserves a mention because it handles the common case of "glue these 1D arrays
together as columns of a table" without any reshaping:

In [101]:
student_ids = np.array([101, 102, 103])
maths = np.array([78, 65, 88])
physics = np.array([82, 70, 94])

table = np.column_stack([student_ids, maths, physics])
print(table)
print("shape:", table.shape)

[[101  78  82]
 [102  65  70]
 [103  88  94]]
shape: (3, 3)


### Splitting

In [102]:
data = np.arange(12)

print("three equal parts:", np.split(data, 3))
print("split at 2 and 7 :", np.split(data, [2, 7]))

grid = np.arange(12).reshape(3, 4)
left, right = np.hsplit(grid, 2)
print("\nleft half:\n", left)
print("right half:\n", right)

three equal parts: [array([0, 1, 2, 3]), array([4, 5, 6, 7]), array([ 8,  9, 10, 11])]
split at 2 and 7 : [array([0, 1]), array([2, 3, 4, 5, 6]), array([ 7,  8,  9, 10, 11])]

left half:
 [[0 1]
 [4 5]
 [8 9]]
right half:
 [[ 2  3]
 [ 6  7]
 [10 11]]


`np.split(arr, n)` cuts into `n` equal pieces and raises an error if it does not divide evenly;
`np.array_split` allows uneven pieces. Passing a list of positions instead of a count cuts at
those boundaries, which is how you carve out a train/test split by index.

## Linear algebra

NumPy is not a full linear algebra package, but `np.linalg` covers everything you need for
ordinary work.

In [103]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
B = np.array([[1.0, 0.0],
              [2.0, 1.0]])

print("element-wise A * B:")
print(A * B)

print("\nmatrix product A @ B:")
print(A @ B)

element-wise A * B:
[[2. 0.]
 [2. 3.]]

matrix product A @ B:
[[4. 1.]
 [7. 3.]]


This is the distinction to be clear about: `*` multiplies matching positions, `@` does true matrix
multiplication (row of the left times column of the right, summed). They are different
operations that happen to share an operator in mathematical notation.

`A @ B` is the modern spelling. `np.matmul(A, B)` is the same thing, and `A.dot(B)` is the older
form you will see in existing code. For 1D arrays, `@` computes the dot product:

In [104]:
prices = np.array([25.0, 40.0, 12.5])
quantities = np.array([4, 2, 10])

print("total bill:", prices @ quantities)
print("same as    :", (prices * quantities).sum())

total bill: 305.0
same as    : 305.0


That is a genuinely good use of the dot product — "multiply pairwise and add it all up" is exactly
what a bill is. `prices @ quantities` says it in one operation.

### Inverse, determinant, and solving systems

In [105]:
print("determinant:", round(np.linalg.det(A), 4))
print("inverse:")
print(np.round(np.linalg.inv(A), 4))
print("\nA @ inv(A) should be the identity:")
print(np.round(A @ np.linalg.inv(A), 10))

determinant: 5.0
inverse:
[[ 0.6 -0.2]
 [-0.2  0.4]]

A @ inv(A) should be the identity:
[[ 1.  0.]
 [-0.  1.]]


Now the practical use. Suppose a canteen sells tea and coffee:

```text
 2 teas + 1 coffee  = ₹110
 1 tea  + 3 coffees = ₹180
```

Written as matrices that is `A @ x = b`, where `x` holds the two unknown prices.

In [106]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
b = np.array([110.0, 180.0])

solution = np.linalg.solve(A, b)
print("tea    :", solution[0])
print("coffee :", solution[1])
print("check  :", A @ solution)

tea    : 30.0
coffee : 50.0
check  : [110. 180.]


`np.linalg.solve(A, b)` is the right tool here. The textbook formula is `x = inv(A) @ b`, and it
gives the same answer, but `solve` is both faster and numerically more stable — it never forms the
inverse, which is the error-prone step. As a rule: if you are about to write `inv(A) @ b`, write
`solve(A, b)` instead.

In [107]:
print("via solve:", np.linalg.solve(A, b))
print("via inv  :", np.linalg.inv(A) @ b)

via solve: [30. 50.]
via inv  : [30. 50.]


When a system has no unique solution, the matrix is *singular* and `inv` refuses:

In [108]:
singular = np.array([[1.0, 2.0],
                     [2.0, 4.0]])
try:
    np.linalg.inv(singular)
except np.linalg.LinAlgError as e:
    print("LinAlgError:", e)

LinAlgError: Singular matrix


The second row here is just the first row doubled, so the two equations carry the same
information and there is not enough to pin down two unknowns. That is what "singular" means —
the determinant is zero:

In [109]:
print("determinant:", np.linalg.det(singular))

determinant: 0.0


A few other functions from `np.linalg` you may meet: `norm` (length of a vector),
`eig` (eigenvalues and eigenvectors), `matrix_rank`, and `lstsq` (least-squares fitting, the
engine behind linear regression).

In [110]:
v = np.array([3.0, 4.0])
print("norm:", np.linalg.norm(v))
print("rank of the singular matrix:", np.linalg.matrix_rank(singular))

norm: 5.0
rank of the singular matrix: 1


## Random numbers, properly

In [111]:
rng = np.random.default_rng(seed=7)

print("uniform 0-1 :", rng.random(3).round(3))
print("integers    :", rng.integers(10, 100, size=5))
print("normal      :", rng.normal(loc=60, scale=12, size=5).round(1))
print("choice      :", rng.choice(["A", "B", "C"], size=6))
print("choice, no repeats:", rng.choice(np.arange(1, 50), size=6, replace=False))

uniform 0-1 : [0.625 0.897 0.776]
integers    : [85 30 14 37 35]
normal      : [60.7 76.1 54.1 52.6 65.9]
choice      : ['C' 'B' 'A' 'C' 'A' 'C']
choice, no repeats: [20 22 24 28 27 25]


`rng.choice` is the sampling workhorse. `replace=False` draws without putting values back — a
lottery draw rather than six independent dice. It also takes a `p=` argument for weighted sampling:

In [112]:
weather = rng.choice(["sunny", "cloudy", "rainy"], size=10, p=[0.5, 0.3, 0.2])
print(weather)

['rainy' 'sunny' 'sunny' 'cloudy' 'sunny' 'sunny' 'cloudy' 'sunny' 'rainy'
 'cloudy']

### Why seeding matters, demonstrated

In [113]:
a = np.random.default_rng(42).integers(0, 100, 5)
b = np.random.default_rng(42).integers(0, 100, 5)
c = np.random.default_rng(99).integers(0, 100, 5)

print("seed 42 :", a)
print("seed 42 :", b, "→ identical")
print("seed 99 :", c, "→ different")

seed 42 : [ 8 77 65 43 43]
seed 42 : [ 8 77 65 43 43] → identical
seed 99 : [95 50 75 56 17] → different


Two generators with the same seed produce the same stream. That is what makes a notebook like this
one possible: the numbers in the mini project below are "random" but they will be the same numbers
for you as for me, so we can talk about the results.

One caution: a seeded generator is reproducible, not predictable-by-eye, and it is not a source of
cryptographic randomness. For passwords and tokens use the `secrets` module instead.

In [114]:
rng = np.random.default_rng(3)
print("first three draws :", rng.integers(0, 10, 3))
print("next three draws  :", rng.integers(0, 10, 3))

first three draws : [8 0 1]
next three draws  : [2 1 8]


Notice that the second call continues the stream rather than restarting it — the generator carries
its state forward. So *where* in your code you create the generator affects the values you get.
Create one at the top and pass it around; re-seeding in the middle of a program is usually a sign
of confusion.

### Shuffling

In [115]:
deck = np.arange(1, 11)
rng = np.random.default_rng(5)

shuffled_copy = rng.permutation(deck)
print("permutation (new array):", shuffled_copy)
print("original untouched     :", deck)

rng.shuffle(deck)
print("after shuffle in place :", deck)

permutation (new array): [ 8  7  2  4  3  5  1 10  6  9]
original untouched     : [ 1  2  3  4  5  6  7  8  9 10]
after shuffle in place : [10  7  4  8  2  1  9  5  6  3]


`permutation` returns a shuffled copy; `shuffle` rearranges in place and returns `None`. Same
distinction as `np.sort` versus `arr.sort()`.

## Vectorisation and performance

The usual claim is "NumPy is faster than loops". That is true often enough to be a good default,
but it is worth knowing why, and when it does not hold.

In [116]:
import time

size = 1_000_000
rng = np.random.default_rng(0)
values = rng.random(size)
values_list = values.tolist()

start = time.perf_counter()
squares_loop = []
for v in values_list:
    squares_loop.append(v * v)
loop_time = time.perf_counter() - start

start = time.perf_counter()
squares_comp = [v * v for v in values_list]
comp_time = time.perf_counter() - start

start = time.perf_counter()
squares_numpy = values ** 2
numpy_time = time.perf_counter() - start

print(f"explicit loop       : {loop_time:.4f} s")
print(f"list comprehension  : {comp_time:.4f} s")
print(f"NumPy vectorised    : {numpy_time:.4f} s")
print(f"NumPy is roughly {loop_time / numpy_time:.0f}x faster than the loop here")

explicit loop       : 0.0695 s
list comprehension  : 0.0475 s
NumPy vectorised    : 0.0029 s
NumPy is roughly 24x faster than the loop here


Your exact numbers will differ from mine — timings depend on the machine, on what else is running,
and on the Python version. The *shape* of the result is the reliable part: for a million elements,
the vectorised version is dramatically faster, and a list comprehension is a modest improvement
over an explicit loop but nowhere near NumPy.

Where does the time go in the loop? Every iteration involves fetching a Python float object,
checking its type, calling its multiplication method, allocating a new float object and appending
it. NumPy does the same million multiplications in a compiled loop over raw doubles, with the type
checked once.

### When vectorisation does not help

Three honest caveats.

**Small arrays.** NumPy has per-call overhead. Below a few dozen elements, a plain Python loop can
win.

In [117]:
small = np.arange(5)
small_list = list(range(5))

start = time.perf_counter()
for _ in range(100_000):
    _ = [v * v for v in small_list]
py_time = time.perf_counter() - start

start = time.perf_counter()
for _ in range(100_000):
    _ = small ** 2
np_time = time.perf_counter() - start

print(f"list comprehension on 5 items: {py_time:.4f} s")
print(f"NumPy on 5 items             : {np_time:.4f} s")

list comprehension on 5 items: 0.0147 s
NumPy on 5 items             : 0.0413 s


On five elements the pure-Python version is usually the faster of the two. None of this matters
for a one-off calculation; it matters if the operation sits inside a loop that runs a million
times, which is exactly when people reach for NumPy and are surprised.

**Memory.** Vectorised code builds whole intermediate arrays. `(a + b) * c - d` on arrays of
10 million floats allocates several 80 MB temporaries. A loop uses almost no extra memory. For
very large data this can be the difference between working and swapping to disk.

**`np.vectorize` is not vectorisation.** The name is misleading:

In [118]:
def grade(score):
    if score >= 80:
        return "A"
    elif score >= 60:
        return "B"
    return "C"


scores = np.array([95, 72, 45, 88, 61])

vectorised_grade = np.vectorize(grade)
print(vectorised_grade(scores))

['A' 'B' 'C' 'A' 'B']


This works and is convenient, but `np.vectorize` is a convenience wrapper that loops in Python
underneath. It buys you broadcasting-compatible syntax, not speed. When performance matters,
express the logic with real array operations:

In [119]:
print(np.select([scores >= 80, scores >= 60], ["A", "B"], default="C"))

['A' 'B' 'C' 'A' 'B']


### The practical rule

Reach for vectorised operations first, because they are usually both faster and shorter. But a
readable loop that runs once on a thousand rows is not a problem worth fixing, and "I made it a
one-liner" is not the same as "I made it better". Measure before optimising, and only optimise the
part that is actually slow.

## Common mistakes, collected

| Mistake | What you see | Fix |
| --- | --- | --- |
| Using `and` / `or` on arrays | `truth value of an array ... is ambiguous` | Use `&`, `\|`, `~` |
| Forgetting parentheses in a compound condition | `ambiguous` error, or nonsense | `(a > 1) & (a < 5)` |
| Wrong `axis` | Plausible but wrong numbers | Check the result's shape against what you expect |
| Assigning a float into an int array | Silent truncation | Create the array with `dtype=float` |
| Modifying a slice, expecting a copy | The original changes too | `.copy()`, or avoid `+=` style operators |
| `x = arr.sort()` | `x` is `None` | `x = np.sort(arr)` |
| Comparing against `np.nan` | Never matches | `np.isnan(arr)` |
| `np.array([[1,2],[3]])` | inhomogeneous shape error | Rows must be equal length |
| Mismatched shapes in arithmetic | `could not be broadcast together` | Print both shapes; often needs `[:, None]` |
| `inv(A) @ b` | Works, but fragile | `np.linalg.solve(A, b)` |

## Debugging: three errors, diagnosed

Reading tracebacks is a skill in itself, so here are three failures with the reasoning spelled out.

### Error 1 — shape mismatch

In [120]:
budget = np.array([[1000, 1200, 900],
                   [1500, 1100, 1300]])
adjustment = np.array([50, 60])

try:
    budget + adjustment
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (2,3) (2,) 


**What Python tells us.** `operands could not be broadcast together with shapes (2,3) (2,)`.

**Why it happened.** Broadcasting aligns from the right. The last axis of `budget` is 3, and
`adjustment` has length 2. Three against two, neither is 1, so there is no way to stretch them.
The length 2 matches `budget`'s *first* axis, which tells us the intent was one adjustment per
row — but NumPy has no way to know that; it only sees shapes.

**Fix.** Say "these are rows" by giving the array a second axis of length 1.

In [121]:
print(budget + adjustment[:, None])

[[1050 1250  950]
 [1560 1160 1360]]


**Better still.** Build it as a column in the first place, so the shape documents the intent:

In [122]:
adjustment_per_row = np.array([[50], [60]])
print(budget + adjustment_per_row)

[[1050 1250  950]
 [1560 1160 1360]]


### Error 2 — the disappearing update

In [123]:
stock = np.array([10, 0, 25, 0, 8])

restocked = stock[stock == 0]
restocked[:] = 50

print("restocked array:", restocked)
print("original stock :", stock)

restocked array: [50 50]
original stock : [10  0 25  0  8]


**What Python tells us.** Nothing. No error, and that is the problem.

**Why it happened.** `stock[stock == 0]` is boolean indexing, which returns a **copy**. We
correctly set both zeros to 50 — in the copy. The original never changed. (Compare with a basic
slice, which *is* a view and would have updated the original. The inconsistency is real, and it is
why `np.shares_memory` is worth knowing.)

**Fix.** Assign through the mask directly, rather than extracting first.

In [124]:
stock = np.array([10, 0, 25, 0, 8])
stock[stock == 0] = 50
print(stock)

[10 50 25 50  8]


The general shape of this bug: if you find yourself pulling elements out, modifying them, and
expecting the source to change, stop. Put the selection on the left of the `=`.

### Error 3 — silent truncation

In [125]:
counts = np.array([7, 12, 5])
totals = np.array([10, 10, 10])

percent_int = counts * 100 // totals
percent_float = counts / totals * 100

print("integer //:", percent_int)
print("float    /:", percent_float)

ratios = np.zeros(3, dtype=int)
ratios[:] = counts / totals
print("stored in an int array:", ratios)

integer //: [ 70 120  50]
float    /: [ 70. 120.  50.]
stored in an int array: [0 1 0]


**What Python tells us.** Again nothing, and again that is the problem.

**Why it happened.** The last case is the dangerous one. `counts / totals` computes
`[0.7, 1.2, 0.5]` perfectly well, but assigning it into an `int64` array truncates each value
towards zero, giving `[0, 1, 0]`. The information is gone with no warning.

**Fix.** Make the destination float, or do not pre-allocate at all — let NumPy choose the dtype
from the operation:

In [126]:
ratios = counts / totals
print(ratios, ratios.dtype)

[0.7 1.2 0.5] float64


The habit to build: whenever you create an array with `np.zeros`, `np.empty` or `np.full` to fill
in later, ask what type the results will be. Counts are integers; averages, ratios and
measurements are floats.

## Mini project — student performance analysis

Time to use the whole toolkit on one problem. A class of 30 students has sat five subjects. We
want the kind of summary a class teacher actually asks for: who is doing well, which subject the
class struggles with, who is at risk of failing, and how each student compares to the class.

The data is generated with a fixed seed so your numbers match mine exactly.

In [127]:
import numpy as np

rng = np.random.default_rng(2024)

subjects = np.array(["Maths", "Physics", "Chemistry", "English", "CompSci"])
first_names = np.array([
    "Aarti", "Bilal", "Chirag", "Divya", "Esha", "Farhan", "Gita", "Harsh",
    "Ishita", "Jatin", "Kavya", "Lokesh", "Manisha", "Nikhil", "Oviya",
    "Pranav", "Qadir", "Rhea", "Sameer", "Tanvi", "Uday", "Vaishali",
    "Wasim", "Xena", "Yash", "Zoya", "Aditya", "Bhavna", "Chetan", "Deepa",
])

n_students = len(first_names)
n_subjects = len(subjects)

# Each subject gets its own difficulty, so the class average differs per subject.
subject_means = np.array([62.0, 58.0, 66.0, 72.0, 69.0])

# Each student has a personal ability offset applied across all subjects.
ability = rng.normal(0, 9, size=(n_students, 1))

noise = rng.normal(0, 7, size=(n_students, n_subjects))

marks = np.clip(np.round(subject_means + ability + noise), 0, 100).astype(int)

print("marks shape:", marks.shape, "→ (students, subjects)")
print(marks[:5])

marks shape: (30, 5) → (students, subjects)
[[72 70 77 93 74]
 [85 70 74 95 81]
 [70 59 62 87 71]
 [59 62 56 55 63]
 [55 44 54 69 65]]


Look at how the data was built, because it is broadcasting doing the work:

- `subject_means` has shape `(5,)` — one baseline per subject, aligning with the columns.
- `ability` has shape `(30, 1)` — one offset per student, aligning with the rows.
- `noise` has shape `(30, 5)` — an independent wobble for every single mark.

Adding all three gives a `(30, 5)` array in one expression. Building it with loops would take
a dozen lines, and this way the structure of the model is visible in the shapes.

`np.clip(..., 0, 100)` keeps marks inside a legal range, `np.round` removes the decimals, and
`.astype(int)` makes them whole numbers, since marks out of 100 are counts, not measurements.

### Step 1 — first look at the data

Before computing anything clever, confirm the data is shaped the way you think it is.

In [128]:
print("students :", marks.shape[0])
print("subjects :", marks.shape[1])
print("dtype    :", marks.dtype)
print("range    :", marks.min(), "to", marks.max())
print("overall average:", marks.mean().round(2))

students : 30
subjects : 5
dtype    : int64
range    : 28 to 96
overall average: 65.62


### Step 2 — per-student and per-subject averages

The whole point of the `axis` argument. One collapses subjects, the other collapses students.

In [129]:
student_average = marks.mean(axis=1)
subject_average = marks.mean(axis=0)

print("student_average shape:", student_average.shape)
print("subject_average shape:", subject_average.shape)

print("\nClass performance by subject")
print("-" * 34)
for subject, avg, sd in zip(subjects, subject_average, marks.std(axis=0, ddof=1)):
    print(f"{subject:10} mean {avg:5.1f}   sd {sd:4.1f}")

student_average shape: (30,)
subject_average shape: (5,)

Class performance by subject
----------------------------------
Maths      mean  62.8   sd  9.8
Physics    mean  58.5   sd 12.3
Chemistry  mean  65.4   sd 12.5
English    mean  72.6   sd 11.7
CompSci    mean  68.7   sd  8.4


The standard deviations are as informative as the means. A subject with a low mean and a small
spread is uniformly hard; a low mean with a large spread suggests the class has split into those
who followed it and those who did not.

### Step 3 — the class table, printed readably

In [130]:
print(f"{'Student':10} " + " ".join(f"{s:>9}" for s in subjects) + f"{'Avg':>8}")
print("-" * 70)

for name, row, avg in zip(first_names, marks, student_average):
    marks_text = " ".join(f"{m:>9}" for m in row)
    print(f"{name:10} {marks_text} {avg:>7.1f}")

Student        Maths   Physics Chemistry   English   CompSci     Avg
----------------------------------------------------------------------
Aarti             72        70        77        93        74    77.2
Bilal             85        70        74        95        81    81.0
Chirag            70        59        62        87        71    69.8
Divya             59        62        56        55        63    59.0
Esha              55        44        54        69        65    57.4
Farhan            68        53        66        77        68    66.4
Gita              65        60        69        75        74    68.6
Harsh             75        57        77        80        75    72.8
Ishita            73        71        96        89        89    83.6
Jatin             73        72        71        74        75    73.0
Kavya             66        69        73        79        76    72.6
Lokesh            64        48        56        72        62    60.4
Manisha           55        43  

Two details in that formatting code:

- `" ".join(f"{m:>9}" for m in row)` builds one string from the row. The part inside `join` is a
  **generator expression** — like a list comprehension without the brackets — which produces each
  formatted number in turn. `>9` right-aligns in a 9-character field.
- Iterating `for name, row, avg in zip(...)` over three sequences at once keeps the three related
  pieces together. Iterating over a 2D array yields its rows, which is why `row` is a whole
  student's marks.

### Step 4 — pass, fail, and distinction counts

Pass mark is 40; distinction is 75 or above.

In [131]:
PASS_MARK = 40
DISTINCTION = 75

failed = marks < PASS_MARK
distinctions = marks >= DISTINCTION

print("total failing marks    :", failed.sum())
print("total distinction marks:", distinctions.sum())
print("\nfailures per subject:")
for subject, count in zip(subjects, failed.sum(axis=0)):
    print(f"  {subject:10} {count}")

students_with_a_fail = failed.any(axis=1)
print("\nstudents failing at least one subject:", students_with_a_fail.sum())
print(first_names[students_with_a_fail])

total failing marks    : 2
total distinction marks: 31

failures per subject:
  Maths      0
  Physics    2
  Chemistry  0
  English    0
  CompSci    0

students failing at least one subject: 2
['Qadir' 'Vaishali']


`failed` is a `(30, 5)` mask. Summing it along `axis=0` counts failures per subject; `any(axis=1)`
asks per student "was there at least one failure anywhere in this row", which gives a `(30,)` mask
we can use to select names.

That last line — `first_names[students_with_a_fail]` — is boolean masking used on a *different*
array than the one the condition came from. It works because the mask's length matches the number
of students. Keeping parallel arrays aligned like this is exactly what Pandas will later do for
you automatically.

### Step 5 — ranking the class

In [132]:
ranking = np.argsort(student_average)[::-1]

print("Top five")
for rank, i in enumerate(ranking[:5], start=1):
    print(f"{rank}. {first_names[i]:10} {student_average[i]:.1f}")

print("\nBottom three")
for rank, i in enumerate(ranking[-3:], start=len(ranking) - 2):
    print(f"{rank}. {first_names[i]:10} {student_average[i]:.1f}")

Top five
1. Ishita     83.6
2. Nikhil     82.0
3. Bilal      81.0
4. Aarti      77.2
5. Jatin      73.0

Bottom three
28. Sameer     51.4
29. Vaishali   49.8
30. Qadir      48.4


`argsort` ascending, reversed, gives positions from best to worst. For the bottom three we take
the last three of that same list and start the rank counter at 28 so the numbering stays truthful.

### Step 6 — subject-wise toppers

Who scored highest in each subject? This is `argmax` along the student axis.

In [133]:
top_per_subject = marks.argmax(axis=0)

for subject, student_index in zip(subjects, top_per_subject):
    print(f"{subject:10} {first_names[student_index]:10} {marks[student_index, subjects == subject][0]}")

Maths      Bilal      85
Physics    Pranav     82
Chemistry  Ishita     96
English    Bilal      95
CompSci    Ishita     89


`marks.argmax(axis=0)` collapses the student axis, so we get one index per subject: the row number
of the best student in that column.

`subjects == subject` inside the index is a small trick worth reading carefully — it builds a
boolean mask over the subject names to pick the right column without needing a counter. Honestly,
`enumerate` would be clearer here:

In [134]:
for j, subject in enumerate(subjects):
    best = marks[:, j].argmax()
    print(f"{subject:10} {first_names[best]:10} {marks[best, j]}")

Maths      Bilal      85
Physics    Pranav     82
Chemistry  Ishita     96
English    Bilal      95
CompSci    Ishita     89


That is the second version being *simpler* than the first, which happens often enough to be worth
pointing out. `enumerate` gives you the column number directly, so you index with `j` and the
intent is obvious. Cleverness that costs readability is not an improvement.

### Step 7 — normalising marks so subjects are comparable

A 70 in English (class mean 72) is not the same achievement as a 70 in Physics (class mean 58).
Converting each mark to a **z-score** — how many standard deviations above or below that subject's
mean — makes them comparable.

In [135]:
z_scores = (marks - marks.mean(axis=0)) / marks.std(axis=0, ddof=1)

print("z-score shape:", z_scores.shape)
print("column means (should be ~0):", z_scores.mean(axis=0).round(10))
print("column sds (should be ~1)  :", z_scores.std(axis=0, ddof=1).round(4))

z-score shape: (30, 5)
column means (should be ~0): [ 0.  0. -0.  0. -0.]
column sds (should be ~1)  : [1. 1. 1. 1. 1.]


One expression, three broadcasts: a `(30, 5)` array minus a `(5,)` array of means, divided by a
`(5,)` array of standard deviations. Both small arrays align with the columns automatically.

Now a fair overall ranking, based on standardised performance rather than raw totals:

In [136]:
overall_z = z_scores.mean(axis=1)
fair_ranking = np.argsort(overall_z)[::-1]

print(f"{'Student':10} {'raw avg':>8} {'z avg':>7} {'raw rank':>9} {'z rank':>7}")
print("-" * 45)

raw_rank_of = np.empty(n_students, dtype=int)
raw_rank_of[np.argsort(student_average)[::-1]] = np.arange(1, n_students + 1)

z_rank_of = np.empty(n_students, dtype=int)
z_rank_of[fair_ranking] = np.arange(1, n_students + 1)

for i in fair_ranking[:8]:
    print(f"{first_names[i]:10} {student_average[i]:8.1f} {overall_z[i]:7.2f} "
          f"{raw_rank_of[i]:9} {z_rank_of[i]:7}")

Student     raw avg   z avg  raw rank  z rank
---------------------------------------------
Ishita         83.6    1.66         1       1
Nikhil         82.0    1.47         2       2
Bilal          81.0    1.45         3       3
Aarti          77.2    1.03         4       4
Jatin          73.0    0.69         5       5
Harsh          72.8    0.68         6       6
Kavya          72.6    0.64         7       7
Pranav         72.0    0.60         8       8


The two lines building `raw_rank_of` deserve unpacking, because inverting a sort order is a genuinely
useful trick and looks cryptic the first time:

```python
raw_rank_of[np.argsort(student_average)[::-1]] = np.arange(1, n_students + 1)
```

- `np.argsort(...)[::-1]` is the list of student indices from best to worst.
- `np.arange(1, 31)` is the ranks 1 to 30.
- Assigning the second into the first *at those positions* says "the best student gets rank 1, the
  second-best gets rank 2, ...". The result is indexed by student, so `raw_rank_of[i]` answers
  "what rank is student `i`" — the inverse of what `argsort` gave us.

Ranks by raw average and by z-score mostly agree, because every student sat the same five subjects.
They diverge for students whose strengths sit in the harder subjects, which is the point of doing it.

### Step 8 — subject correlations

Do students who do well in Maths also do well in Physics?

In [137]:
correlations = np.corrcoef(marks, rowvar=False)

print(f"{'':10}" + "".join(f"{s:>10}" for s in subjects))
for subject, row in zip(subjects, correlations):
    print(f"{subject:10}" + "".join(f"{value:>10.2f}" for value in row))

               Maths   Physics Chemistry   English   CompSci
Maths           1.00      0.58      0.67      0.63      0.68
Physics         0.58      1.00      0.60      0.52      0.66
Chemistry       0.67      0.60      1.00      0.60      0.68
English         0.63      0.52      0.60      1.00      0.73
CompSci         0.68      0.66      0.68      0.73      1.00


`np.corrcoef` computes correlation coefficients between -1 and 1. `rowvar=False` tells it that our
**columns** are the variables and the rows are observations — without it, NumPy would correlate the
30 students with each other, which is not the question. Getting `rowvar` wrong is a classic
mistake, and the giveaway is the shape of the result: 5×5 is what we want, 30×30 is not.

The diagonal is 1.0 because a subject correlates perfectly with itself. Everything is fairly
strongly positive, which the data generator guarantees: every student's marks share that single
`ability` offset. Real data is rarely this tidy.

### Step 9 — a compact summary

In [138]:
def summarise(marks, names, subjects, pass_mark=40):
    """Return the numbers a class summary needs, without printing anything."""
    per_student = marks.mean(axis=1)
    per_subject = marks.mean(axis=0)
    return {
        "class_average": marks.mean(),
        "hardest_subject": subjects[per_subject.argmin()],
        "easiest_subject": subjects[per_subject.argmax()],
        "top_student": names[per_student.argmax()],
        "top_average": per_student.max(),
        "students_at_risk": names[(marks < pass_mark).any(axis=1)],
        "pass_rate": (marks >= pass_mark).mean(),
    }


summary = summarise(marks, first_names, subjects)

for key, value in summary.items():
    if isinstance(value, np.ndarray):
        print(f"{key:18}: {', '.join(value)}")
    elif isinstance(value, (float, np.floating)):
        print(f"{key:18}: {value:.3f}")
    else:
        print(f"{key:18}: {value}")

class_average     : 65.620
hardest_subject   : Physics
easiest_subject   : English
top_student       : Ishita
top_average       : 83.600
students_at_risk  : Qadir, Vaishali
pass_rate         : 0.987


Returning a dictionary rather than printing inside the function is a habit worth forming. The
function computes, the caller decides what to do with the result — print it, write it to a file,
compare two classes. A function that prints is a function you can only use one way.

### What to take from the project

Every step used the same handful of ideas:

- `axis` to choose whether a question is about students or subjects
- broadcasting to combine arrays of different shapes without loops
- boolean masks to count, filter and select across parallel arrays
- `argsort` / `argmax` to answer "which one", not just "what value"

One limitation is worth naming. We kept names in one array, marks in another, and subjects in a
third, and every operation depended on those staying aligned. Nothing enforces that. Delete a
student from `marks` and forget `first_names` and every result silently becomes wrong. That is
precisely the problem Pandas solves by attaching labels to the data, which is where the next
notebook picks up.

## Where NumPy sits

```text
NumPy          fast typed arrays and the maths on them
   ↓
Pandas         the same arrays, with row and column labels attached
   ↓
Matplotlib     turns arrays into plots
```

Pandas columns *are* NumPy arrays — `df["salary"].to_numpy()` hands you one directly — and
Matplotlib accepts arrays wherever it accepts data. Nothing you learned here is left behind when
you move on; it becomes the layer underneath.

## Solutions

Read these after attempting the exercises. Each one explains the reasoning, not just the answer.

### Arrays and creation

In [139]:
# Level 1
evens = np.arange(2, 21, 2)
print(evens, evens.shape, evens.dtype, evens.sum())

[ 2  4  6  8 10 12 14 16 18 20] (10,) int64 110


`np.arange(2, 21, 2)` starts at 2, steps by 2, and stops *before* 21 — so 20 is included. Writing
`np.arange(2, 20, 2)` is the classic off-by-one here and gives only nine values.

In [140]:
# Level 2
sevens = np.full((3, 5), 7)
print(sevens)

grid = np.arange(1, 16).reshape(3, 5)
print(grid)

[[7 7 7 7 7]
 [7 7 7 7 7]
 [7 7 7 7 7]]
[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]]


`np.full(shape, value)` is the direct way. `np.ones((3, 5)) * 7` also works but gives floats.
For the second part, `arange` produces 15 numbers and `reshape(3, 5)` lays them out row by row,
which happens to be exactly the pattern asked for.

In [141]:
# Level 3
lin = np.linspace(0, 1, 11)
ara = np.arange(0, 1.1, 0.1)

print("linspace:", lin)
print("arange  :", ara)
print("lengths :", len(lin), len(ara))
print("identical?", np.array_equal(lin, ara))

# Shift the range slightly and the agreement disappears.
lin2 = np.linspace(0, 0.6, 7)
ara2 = np.arange(0, 0.7, 0.1)
print("\nshifted range identical?", np.array_equal(lin2, ara2))
print("difference:", lin2 - ara2)

linspace: [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
arange  : [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
lengths : 11 11
identical? True

shifted range identical? False
difference: [ 0.00000000e+00 -1.38777878e-17 -2.77555756e-17 -5.55111512e-17
 -5.55111512e-17 -5.55111512e-17 -1.11022302e-16]


On this particular range the two agree exactly — 11 values, all identical — so the first part of
the exercise is a bit of a trap. `arange` is not *always* wrong with fractional steps.

Move the range slightly, though, and they diverge: `np.arange(0, 0.7, 0.1)` produces
0.6000000000000001 where `linspace` produces exactly 0.6. `arange` has to derive each value from
the step, and 0.1 is not exactly representable in binary. `linspace` works from the two endpoints
and the count, so its values are as close as floating point allows and the count is guaranteed to
be what you asked for.

For a plot axis, use `linspace`. What you want is "200 points from 0 to 2π", and that is exactly
what `linspace(0, 2*np.pi, 200)` says. With `arange` you would compute the step yourself and then
hope the endpoint lands where you expect.

### Indexing, views and reshaping

In [142]:
# Level 1
temps = np.array([18, 21, 25, 30, 28, 22, 19])
print("first three :", temps[:3])
print("last two    :", temps[-2:])
print("every second:", temps[::2])
print("reversed    :", temps[::-1])

first three : [18 21 25]
last two    : [22 19]
every second: [18 25 28 19]
reversed    : [19 22 28 30 25 21 18]


In [143]:
# Level 2
table = np.arange(1, 21).reshape(4, 5)
print(table)
print("\nsecond row      :", table[1])
print("last column     :", table[:, -1])
print("bottom-right 2x2:\n", table[-2:, -2:])
print("rows 0 and 2:\n", table[[0, 2]])

[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]
 [16 17 18 19 20]]

second row      : [ 6  7  8  9 10]
last column     : [ 5 10 15 20]
bottom-right 2x2:
 [[14 15]
 [19 20]]
rows 0 and 2:
 [[ 1  2  3  4  5]
 [11 12 13 14 15]]


`table[-2:, -2:]` uses negative slices on both axes: "the last two rows, and within them the last
two columns". Counting from the end avoids hard-coding the size, so the same expression works on
any table with at least two rows and columns.

`table[[0, 2]]` is fancy indexing with a list of row numbers. `table[0:3:2]` gives the same rows
here via a step, but the list form states which rows you want rather than encoding it in
arithmetic.

In [144]:
# Level 3
def top_row_sum(matrix):
    row_totals = matrix.sum(axis=1)
    return row_totals.max()


def top_row_sum_destructive(matrix):
    """Deliberately broken: normalises the best row in place before summing."""
    best = matrix[matrix.sum(axis=1).argmax()]
    best -= best.min()
    return best.sum()


data = np.array([[1.0, 2.0, 3.0], [10.0, 20.0, 30.0], [4.0, 5.0, 6.0]])

print("safe version  :", top_row_sum(data))
print("input after   :\n", data)

row_view = data[1]
print("\nrow 1 is a view:", np.shares_memory(data, row_view))

print("destructive version:", top_row_sum_destructive(data))
print("input after:\n", data)

safe version  : 60.0
input after   :
 [[ 1.  2.  3.]
 [10. 20. 30.]
 [ 4.  5.  6.]]

row 1 is a view: True
destructive version: 30.0
input after:
 [[ 1.  2.  3.]
 [ 0. 10. 20.]
 [ 4.  5.  6.]]


The safe version never indexes into a single row; `matrix.sum(axis=1)` builds a new array of
totals, and `.max()` reduces that to a number. Nothing points back at the caller's data.

The destructive version grabs `matrix[i]`, which is a **view** on row `i`, and then uses `-=`,
which writes in place. The caller's array is modified as a side effect. `np.shares_memory` proves
the row and the table share a buffer.

The lesson is not "never use `-=`" but "know whether the thing on the left owns its memory".

### Maths, axis and broadcasting

In [145]:
# Level 1
prices = np.array([250, 480, 120, 890, 310])
print("total      :", prices.sum())
print("mean       :", prices.mean())
print("most expensive:", prices.max())
print("cheapest at position", prices.argmin(), "→", prices[prices.argmin()])

total      : 2050
mean       : 410.0
most expensive: 890
cheapest at position 2 → 120


In [146]:
# Level 2
sales = np.array([[120, 135, 150, 160],
                  [ 90,  85, 110, 100],
                  [200, 210, 190, 230]])
products = np.array(["Keyboard", "Mouse", "Monitor"])
quarters = np.array(["Q1", "Q2", "Q3", "Q4"])

best_quarter_index = sales.argmax(axis=1)
for product, q in zip(products, best_quarter_index):
    print(f"{product:9} peaked in {quarters[q]}")

annual = sales.sum(axis=1)
print("\nbest-selling product:", products[annual.argmax()], "with", annual.max())

Keyboard  peaked in Q4
Mouse     peaked in Q3
Monitor   peaked in Q4

best-selling product: Monitor with 830


`sales.argmax(axis=1)` collapses the quarter axis, leaving one index per product — the position of
that product's best quarter. Using `axis=0` instead would answer a different question ("which
product was best in each quarter") and return four values instead of three. The length of the
result is the quickest check that you picked the right axis.

In [147]:
# Level 3
distances = np.array([5.0, 12.5, 3.2, 8.8])
fuel_rates = np.array([0.08, 0.11, 0.15])

fuel_needed = fuel_rates[:, None] * distances

print("shape:", fuel_needed.shape, "→ (vehicles, trips)")
print(np.round(fuel_needed, 3))

shape: (3, 4) → (vehicles, trips)
[[0.4   1.    0.256 0.704]
 [0.55  1.375 0.352 0.968]
 [0.75  1.875 0.48  1.32 ]]


The result must be `(3, 4)`: three vehicles, four trips, one number for every combination. So the
vehicle array has to become a column, `(3, 1)`, and the trip array stays a row, `(4,)` → `(1, 4)`.
Broadcasting then stretches both to `(3, 4)`.

Writing `fuel_rates * distances` without the `[:, None]` fails, because `(3,)` against `(4,)`
cannot align. That error message is a useful signal: it means you asked for pairwise combination
when you meant every-combination.

## Quick reference

**Creating**

```python
np.array([1, 2, 3])            np.zeros((2, 3))        np.ones(5, dtype=int)
np.full((2, 2), 7)             np.arange(0, 10, 2)     np.linspace(0, 1, 5)
np.eye(3)                      np.zeros_like(other)    rng = np.random.default_rng(42)
```

**Inspecting**

```python
a.shape    a.ndim    a.size    a.dtype    a.nbytes    np.shares_memory(a, b)
```

**Indexing**

```python
a[0]              a[-1]            a[1:4]         a[::2]        a[::-1]
a[row, col]       a[row, :]        a[:, col]      a[1:3, 2:]
a[[0, 2, 4]]      a[a > 10]        a[(a > 1) & (a < 5)]
```

**Shape**

```python
a.reshape(3, 4)   a.reshape(-1, 1)   a.ravel()   a.flatten()   a.T
a[:, None]        np.expand_dims(a, 1)           np.squeeze(a)
```

**Maths and aggregation**

```python
a + b   a * b   a @ b   np.sqrt(a)   np.round(a, 2)   np.abs(a)
a.sum(axis=0)   a.mean(axis=1)   np.median(a)   a.std(ddof=1)
a.min()   a.max()   a.argmin()   a.argmax()   np.ptp(a)
np.cumsum(a)   np.nanmean(a)   a.sum(axis=1, keepdims=True)
```

**Selection and logic**

```python
np.where(cond, x, y)      np.select([c1, c2], [v1, v2], default=v3)
mask.sum()   mask.mean()   mask.any()   mask.all()   ~mask
np.isnan(a)   np.isin(a, values)   np.clip(a, lo, hi)
```

**Sorting**

```python
np.sort(a)   np.argsort(a)   np.unique(a, return_counts=True)   np.searchsorted(a, v)
```

**Combining**

```python
np.concatenate([a, b], axis=0)   np.vstack([a, b])   np.hstack([a, b])
np.stack([a, b], axis=1)         np.column_stack([a, b])   np.split(a, 3)
```

**Linear algebra**

```python
a @ b   np.linalg.solve(A, b)   np.linalg.inv(A)   np.linalg.det(A)
np.linalg.norm(v)   np.corrcoef(data, rowvar=False)
```

**Random**

```python
rng = np.random.default_rng(42)
rng.random(n)   rng.integers(lo, hi, size)   rng.normal(mean, sd, size)
rng.choice(values, size, replace=False, p=weights)   rng.permutation(a)
```

### Where to go next

Open `02_Pandas.ipynb`. Pandas takes these arrays, attaches labels to the rows and columns, and
adds the tools for data that is messy, mixed-type and real. Almost every NumPy idea here —
boolean masks, `axis`, broadcasting, vectorised thinking — reappears there wearing different clothes.